<a href="https://colab.research.google.com/github/keylanami/mcdm-entropy-saw/blob/main/Dynamic_Fare_Master_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import pandas as pd
import requests
import numpy as np
import geopandas as gpd
import rasterio

from shapely.geometry import (
    Point,
    Polygon,
    MultiPolygon,
    shape,
    mapping
)

from shapely.ops import unary_union
from rasterio.mask import mask
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================
# 1. LOAD LAYER A — DAILY MULTIMODAL DEMAND
# ============================================================

df_demand = pd.read_csv(
    'Filedata Jumlah Penumpang Angkutan Umum yang Terlayani per Hari Tahun 2023.csv'
)

# Standardize mode names
df_demand['jenis_moda'] = (
    df_demand['jenis_moda']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Keep only the 3 primary modes
df_master = df_demand[
    df_demand['jenis_moda'].isin([
        'transjakarta',
        'mrt',
        'lrt'
    ])
].copy()

# ------------------------------------------------------------
# Clean date
# ------------------------------------------------------------

df_master['tanggal'] = (
    df_master['tanggal']
    .astype(str)
    .str.strip()
)

df_master = df_master[
    df_master['tanggal'] != ''
].copy()

df_master['date'] = pd.to_datetime(
    df_master['tanggal'],
    format='%Y-%m-%d',
    errors='coerce'
)

df_master = df_master.dropna(
    subset=['date']
).copy()

# ------------------------------------------------------------
# Temporal variables
# ------------------------------------------------------------

df_master['day_of_week'] = (
    df_master['date'].dt.dayofweek
)

df_master['is_weekend'] = (
    df_master['day_of_week'].isin([5, 6])
)

# ------------------------------------------------------------
# Rename columns
# ------------------------------------------------------------

df_master = df_master.rename(
    columns={
        'jenis_moda': 'mode',
        'jumlah_penumpang_per_hari': 'passenger_count'
    }
)

# ------------------------------------------------------------
# Clean passenger count
#
# "98,351" -> 98351
# "2,372"  -> 2372
# ------------------------------------------------------------

df_master['passenger_count'] = (
    df_master['passenger_count']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df_master['passenger_count'] = pd.to_numeric(
    df_master['passenger_count'],
    errors='coerce'
)

print("=== PASSENGER COUNT VALIDATION ===")

print(
    f"Missing passenger_count: "
    f"{df_master['passenger_count'].isna().sum()}"
)

print(
    f"Data type: "
    f"{df_master['passenger_count'].dtype}"
)



# ============================================================
# CONFIGURATION
# ============================================================

PRIMARY_RADIUS_M = 500
SENSITIVITY_RADII_M = [400, 500, 800]

METRIC_CRS = "EPSG:32748"
WGS84_CRS = "EPSG:4326"

BIG_CITY_QUERY_URL = (
    "https://geoservices.big.go.id/gis/rest/services/"
    "DISIGT/BatasWilayah/FeatureServer/0/query"
)

BOUNDARY_CACHE = (
    "dki_mainland_boundary_cache_utm48.geojson"
)

WORLDPOP_URL = (
    "https://data.worldpop.org/GIS/Population/Global_2015_2030/"
    "R2025A/2023/IDN/v1/100m/constrained/"
    "idn_pop_2023_CN_100m_R2025A_v1.tif"
)

WORLDPOP_PATH = (
    "idn_pop_2023_CN_100m_R2025A_v1.tif"
)


# ============================================================
# 2A. LOAD GIS DATASETS
# ============================================================

GIS_ID26_PATH = (
    "Query Titik_Transportasi_Umum_Jakarta_v3 (ID 26).txt"
)

GIS_LRT_PATH = (
    "Query Titik_LRT_2026.txt"
)


# ------------------------------------------------------------
# LOAD ID26
# Used for:
#   - TransJakarta
#   - MRT Jakarta
# ------------------------------------------------------------

with open(
    GIS_ID26_PATH,
    "r",
    encoding="utf-8"
) as f:

    gis_id26 = json.load(f)


# ------------------------------------------------------------
# LOAD LRT 2026
# The LRT file is a JSON LIST containing station records.
# ------------------------------------------------------------

with open(
    GIS_LRT_PATH,
    "r",
    encoding="utf-8"
) as f:

    gis_lrt = json.load(f)


# ============================================================
# 2B. BUILD TRANSPORT POINTS
# ============================================================

transport_rows = []


# ------------------------------------------------------------
# SOURCE 1: ID26
# ------------------------------------------------------------

ID26_FUNCTIONS = {
    "transjakarta": "HALTE TRANSJAKARTA",
    "mrt": "STASIUN MRT - FASE 1"
}


for feature in gis_id26.get(
    "features",
    []
):

    attributes = feature.get(
        "attributes",
        {}
    )

    geometry = feature.get(
        "geometry",
        {}
    )

    fungsi = str(
        attributes.get(
            "fungsi",
            ""
        )
    ).strip().upper()


    mode = None

    for mode_name, fungsi_target in ID26_FUNCTIONS.items():

        if fungsi == fungsi_target:

            mode = mode_name
            break


    if mode is None:
        continue


    x = geometry.get("x")
    y = geometry.get("y")


    if x is None or y is None:
        continue


    try:

        x = float(x)
        y = float(y)

    except (
        TypeError,
        ValueError
    ):

        continue


    transport_rows.append({

        "mode": mode,

        "source_dataset": "ID26",

        "station_name":
            attributes.get("nama"),

        "geometry":
            Point(x, y)

    })


# ------------------------------------------------------------
# SOURCE 2: LRT 2026
# ------------------------------------------------------------
#
# Query Titik_LRT_2026.txt has this structure:
#
# [
#   {
#       "id": 1,
#       "station_name": "...",
#       "phase": "...",
#       "latitude": ...,
#       "longitude": ...
#   },
#   ...
# ]
#
# ------------------------------------------------------------

if not isinstance(
    gis_lrt,
    list
):

    raise ValueError(
        "Query Titik_LRT_2026.txt "
        "must contain a JSON list of stations."
    )


for station in gis_lrt:

    if not isinstance(
        station,
        dict
    ):

        continue


    latitude = station.get(
        "latitude"
    )

    longitude = station.get(
        "longitude"
    )


    if latitude is None or longitude is None:
        continue


    try:

        latitude = float(latitude)
        longitude = float(longitude)

    except (
        TypeError,
        ValueError
    ):

        continue


    transport_rows.append({

        "mode": "lrt",

        "source_dataset":
            "Query Titik_LRT_2026",

        "station_name":
            station.get("station_name"),

        "phase":
            station.get("phase"),

        "geometry":
            Point(
                longitude,
                latitude
            )

    })


# ============================================================
# 2C. CREATE TRANSPORT GEODATAFRAME
# ============================================================

gdf_transport = gpd.GeoDataFrame(

    transport_rows,

    geometry="geometry",

    crs=WGS84_CRS

)


if gdf_transport.empty:

    raise ValueError(
        "No valid transport GIS points were found."
    )


# ============================================================
# 2D. REMOVE DUPLICATES
# ============================================================

gdf_transport["coord_x"] = (
    gdf_transport
    .geometry
    .x
    .round(7)
)

gdf_transport["coord_y"] = (
    gdf_transport
    .geometry
    .y
    .round(7)
)


gdf_transport = (
    gdf_transport

    .drop_duplicates(
        subset=[
            "mode",
            "coord_x",
            "coord_y"
        ]
    )

    .copy()
)


# ============================================================
# 2D. VALIDATE EXPECTED COUNTS
# ============================================================

EXPECTED_COUNTS = {

    "transjakarta": 378,

    "mrt": 22,

    "lrt": 11

}


actual_counts = (
    gdf_transport["mode"]
    .value_counts()
    .to_dict()
)


print(
    "=== TRANSPORT GIS POINT VALIDATION ==="
)


for mode, expected in EXPECTED_COUNTS.items():

    actual = actual_counts.get(
        mode,
        0
    )


    print(
        f"{mode}: "
        f"{actual} points "
        f"(expected {expected})"
    )


    if actual != expected:

        raise ValueError(

            f"Invalid GIS point count "
            f"for {mode}: "

            f"got {actual}, "
            f"expected {expected}."

        )


print(
    "\n✓ All transport GIS point counts "
    "match the expected dataset."
)

# ============================================================
# 2E. BUILD MODE-SPECIFIC TRANSPORT POINTS
# ============================================================
#
# Create one GeoDataFrame for each transport mode.
#
# These mode-specific point sets are used later to create
# accessibility buffers for C2 Population Access Coverage.
# ============================================================

mode_points = {}

for mode in EXPECTED_COUNTS.keys():

    points = (
        gdf_transport[
            gdf_transport["mode"] == mode
        ]
        .copy()
    )

    if points.empty:

        raise ValueError(
            f"No transport GIS points found "
            f"for mode: {mode}"
        )

    expected_count = EXPECTED_COUNTS[mode]

    actual_count = len(points)

    if actual_count != expected_count:

        raise ValueError(
            f"Mode point count mismatch for "
            f"{mode}: "
            f"got {actual_count}, "
            f"expected {expected_count}."
        )

    mode_points[mode] = points


print("\n=== MODE-SPECIFIC GIS POINTS ===")

for mode, points in mode_points.items():

    print(
        f"{mode}: "
        f"{len(points)} points"
    )

print(
    "✓ Mode-specific transport point sets created."
)

# ============================================================
# 2E. LOAD / REBUILD MAINLAND DKI BOUNDARY
# ============================================================

from numbers import Number
from shapely.validation import make_valid


# ------------------------------------------------------------
# Helper:
# Remove Z/M values and invalid coordinate values from
# GeoJSON coordinates.
#
# Example:
# [106.80, -6.20, None, None]
# ->
# [106.80, -6.20]
# ------------------------------------------------------------

def clean_geojson_coordinates(
    coordinates
):

    # --------------------------------------------------------
    # Base case: this is one coordinate pair/triplet.
    # --------------------------------------------------------

    if (
        isinstance(coordinates, list)
        and len(coordinates) >= 2
        and isinstance(coordinates[0], Number)
        and isinstance(coordinates[1], Number)
    ):

        x = float(
            coordinates[0]
        )

        y = float(
            coordinates[1]
        )


        if not (
            np.isfinite(x)
            and np.isfinite(y)
        ):

            raise ValueError(
                "Coordinate contains "
                "non-finite X/Y values."
            )


        return [
            x,
            y
        ]


    # --------------------------------------------------------
    # Recursive case: nested coordinate arrays.
    # --------------------------------------------------------

    if isinstance(
        coordinates,
        list
    ):

        cleaned = []


        for item in coordinates:

            try:

                cleaned.append(
                    clean_geojson_coordinates(
                        item
                    )
                )

            except (
                TypeError,
                ValueError
            ):

                # Ignore malformed nested coordinate
                # entries rather than crashing the
                # complete Jakarta boundary query.

                continue


        return cleaned


    raise TypeError(
        "Unsupported GeoJSON coordinate structure."
    )


# ------------------------------------------------------------
# Helper:
# Clean one GeoJSON geometry.
# ------------------------------------------------------------

def clean_geojson_geometry(
    geometry
):

    if geometry is None:

        return None


    geometry_type = geometry.get(
        "type"
    )

    coordinates = geometry.get(
        "coordinates"
    )


    if (
        geometry_type is None
        or coordinates is None
    ):

        raise ValueError(
            "GeoJSON geometry is missing "
            "type or coordinates."
        )


    cleaned_coordinates = (
        clean_geojson_coordinates(
            coordinates
        )
    )


    cleaned_geometry = {

        "type":
            geometry_type,

        "coordinates":
            cleaned_coordinates

    }


    return cleaned_geometry


# ============================================================
# TRY EXISTING CACHE FIRST
# ============================================================

use_cached_boundary = False


if os.path.exists(
    BOUNDARY_CACHE
):

    print(
        "\nLoading cached mainland DKI boundary..."
    )


    try:

        with open(
            BOUNDARY_CACHE,
            "r",
            encoding="utf-8"
        ) as f:

            boundary_payload = json.load(f)


        boundary_features = (
            boundary_payload.get(
                "features",
                []
            )
        )


        if not boundary_features:

            raise ValueError(
                "Boundary cache contains no features."
            )


        cache_rows = []


        for feature in boundary_features:

            geometry = feature.get(
                "geometry"
            )


            if geometry is None:
                continue


            try:

                polygon = shape(
                    geometry
                )

            except Exception:

                continue


            if polygon.is_empty:
                continue


            if not polygon.is_valid:

                polygon = make_valid(
                    polygon
                )


            if polygon.is_empty:
                continue


            cache_rows.append({

                "properties_text":
                    "DKI JAKARTA MAINLAND",

                "geometry":
                    polygon

            })


        if not cache_rows:

            raise ValueError(
                "Boundary cache contains "
                "no valid geometries."
            )


        mainland = gpd.GeoDataFrame(

            cache_rows,

            geometry="geometry",

            crs=METRIC_CRS

        )


        cached_union = unary_union(
            mainland.geometry
        )


        cached_area_km2 = (
            cached_union.area
            / 1_000_000
        )


        # ----------------------------------------------------
        # Validate cache before using it.
        # ----------------------------------------------------

        if not (
            600
            <= cached_area_km2
            <= 700
        ):

            raise ValueError(
                "Cached mainland boundary "
                "failed area sanity check."
            )


        use_cached_boundary = True


        print(
            f"✓ Cached boundary validated: "
            f"{cached_area_km2:,.2f} km²"
        )


    except Exception as cache_error:

        print(
            "\nCached boundary is invalid or unreadable."
        )

        print(
            f"Reason: {cache_error}"
        )

        print(
            "Rebuilding boundary from BIG service..."
        )


        use_cached_boundary = False


# ============================================================
# REBUILD BOUNDARY WHEN CACHE IS NOT AVAILABLE
# ============================================================

if not use_cached_boundary:

    print(
        "\nBoundary cache not found or invalid."
    )

    print(
        "Rebuilding mainland DKI boundary "
        "from BIG service..."
    )


    # --------------------------------------------------------
    # Jakarta bounding box.
    #
    # Restrict the query spatially so BIG does not have to
    # process the full national administrative layer.
    # --------------------------------------------------------

    JAKARTA_BBOX = (
        "106.60,-6.45,107.05,-6.00"
    )


    params = {

        "where":
            "1=1",

        "geometry":
            JAKARTA_BBOX,

        "geometryType":
            "esriGeometryEnvelope",

        "inSR":
            "4326",

        "spatialRel":
            "esriSpatialRelIntersects",

        "outFields":
            "namobj",

        "returnGeometry":
            "true",

        "returnZ":
            "false",

        "returnM":
            "false",

        "outSR":
            "4326",

        "resultRecordCount":
            100,

        "f":
            "geojson"

    }


    # --------------------------------------------------------
    # HTTP session with retry.
    # --------------------------------------------------------

    session = requests.Session()


    retry_strategy = Retry(

        total=3,

        connect=3,

        read=3,

        backoff_factor=2,

        status_forcelist=[
            429,
            500,
            502,
            503,
            504
        ],

        allowed_methods=[
            "GET"
        ]

    )


    adapter = HTTPAdapter(
        max_retries=retry_strategy
    )


    session.mount(
        "https://",
        adapter
    )


    response = session.get(

        BIG_CITY_QUERY_URL,

        params=params,

        timeout=(
            30,
            180
        )

    )


    response.raise_for_status()


    payload = response.json()


    # --------------------------------------------------------
    # Validate server response.
    # --------------------------------------------------------

    if "error" in payload:

        raise ValueError(

            "BIG boundary service returned an error:\n"

            f"{payload['error']}"

        )


    if (
        payload.get("type")
        != "FeatureCollection"
    ):

        raise ValueError(

            "Unexpected BIG response format: "

            f"{str(payload)[:500]}"

        )


    raw_features = payload.get(
        "features",
        []
    )


    print(
        f"BIG returned "
        f"{len(raw_features)} features "
        f"inside the Jakarta bounding box."
    )


    if not raw_features:

        raise ValueError(
            "BIG returned no boundary features."
        )


    # ========================================================
    # CLEAN FEATURES ONE BY ONE
    # ========================================================

    boundary_rows = []

    invalid_features = []


    for feature_index, feature in enumerate(
        raw_features
    ):

        properties = feature.get(
            "properties",
            {}
        )

        geometry = feature.get(
            "geometry"
        )


        if geometry is None:

            invalid_features.append(
                feature_index
            )

            continue


        try:

            # ------------------------------------------------
            # Remove Z/M/null coordinate values first.
            # ------------------------------------------------

            cleaned_geometry = (
                clean_geojson_geometry(
                    geometry
                )
            )


            polygon = shape(
                cleaned_geometry
            )


            if polygon.is_empty:

                invalid_features.append(
                    feature_index
                )

                continue


            # ------------------------------------------------
            # Repair invalid polygon topology if needed.
            # ------------------------------------------------

            if not polygon.is_valid:

                polygon = make_valid(
                    polygon
                )


            if polygon.is_empty:

                invalid_features.append(
                    feature_index
                )

                continue


            namobj = str(
                properties.get(
                    "namobj",
                    ""
                )
            ).strip()


            boundary_rows.append({

                "namobj":
                    namobj,

                "geometry":
                    polygon

            })


        except Exception as error:

            invalid_features.append(
                feature_index
            )

            print(

                f"Warning: feature "
                f"{feature_index} skipped: "
                f"{error}"

            )


    # --------------------------------------------------------
    # Validate cleaned geometries.
    # --------------------------------------------------------

    if not boundary_rows:

        raise ValueError(

            "No valid BIG boundary geometries "
            "could be reconstructed."

        )


    boundary = gpd.GeoDataFrame(

        boundary_rows,

        geometry="geometry",

        crs=WGS84_CRS

    )


    if boundary.empty:

        raise ValueError(
            "Cleaned BIG boundary is empty."
        )


    # ========================================================
    # STANDARDIZE NAMES
    # ========================================================

    boundary["namobj_clean"] = (

        boundary["namobj"]

        .astype(str)

        .str.strip()

        .str.upper()

    )


    print(
        "\nBIG administrative names returned:"
    )

    print(
        boundary[
            "namobj_clean"
        ]
        .value_counts()
        .to_string()
    )


    # ========================================================
    # SELECT JAKARTA MAINLAND ADMINISTRATIVE AREAS
    # ========================================================

    jakarta = boundary[
        boundary[
            "namobj_clean"
        ].str.contains(
            "JAKARTA",
            na=False
        )
    ].copy()


    if jakarta.empty:

        raise ValueError(

            "No Jakarta administrative polygons "
            "were found in BIG response."

        )


    # --------------------------------------------------------
    # Exclude Kepulauan Seribu.
    # -------------------------------------------------------

    mainland = jakarta[
        ~jakarta[
            "namobj_clean"
        ].str.contains(
            "KEPULAUAN SERIBU",
            na=False
        )
    ].copy()


    if mainland.empty:

        raise ValueError(
            "Mainland Jakarta boundary is empty."
        )


    # ========================================================
    # CONVERT TO METRIC CRS
    # ========================================================

    mainland = (
        mainland
        .to_crs(METRIC_CRS)
        .copy()
    )


    # ========================================================
    # UNION MAINLAND POLYGONS
    # ========================================================

    mainland_union_temp = unary_union(
        mainland.geometry
    )


    if mainland_union_temp.is_empty:

        raise ValueError(
            "Unified mainland geometry is empty."
        )


    # --------------------------------------------------------
    # Repair union if necessary.
    # --------------------------------------------------------

    if not mainland_union_temp.is_valid:

        mainland_union_temp = make_valid(
            mainland_union_temp
        )


    if mainland_union_temp.is_empty:

        raise ValueError(
            "Repaired mainland geometry is empty."
        )


    # ========================================================
    # SAVE CLEANED CACHE
    # ========================================================

    cache_payload = {

        "type":
            "FeatureCollection",

        "features": [

            {

                "type":
                    "Feature",

                "properties": {

                    "source":
                        "BIG DISIGT BatasWilayah",

                    "crs":
                        METRIC_CRS,

                    "scope":
                        "DKI Jakarta mainland",

                    "kepulauan_seribu_excluded":
                        True

                },

                "geometry":
                    mapping(
                        mainland_union_temp
                    )

            }

        ]

    }


    with open(
        BOUNDARY_CACHE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            cache_payload,
            f
        )


    print(
        f"\n✓ Clean boundary cache created: "
        f"{BOUNDARY_CACHE}"
    )


    # --------------------------------------------------------
    # Use generated geometry directly.
    # --------------------------------------------------------

    mainland = gpd.GeoDataFrame(

        [

            {

                "properties_text":
                    "DKI JAKARTA MAINLAND",

                "geometry":
                    mainland_union_temp

            }

        ],

        geometry="geometry",

        crs=METRIC_CRS

    )


# ============================================================
# FINAL BOUNDARY VALIDATION
# ============================================================

mainland_union = unary_union(
    mainland.geometry
)


if mainland_union.is_empty:

    raise ValueError(
        "Mainland unified geometry is empty."
    )


if not mainland_union.is_valid:

    mainland_union = make_valid(
        mainland_union
    )


mainland_area_km2 = (
    mainland_union.area
    / 1_000_000
)


print(
    "\n=== DKI MAINLAND BOUNDARY ==="
)

print(
    f"Mainland area: "
    f"{mainland_area_km2:,.2f} km²"
)


if not (
    600
    <= mainland_area_km2
    <= 700
):

    raise ValueError(

        "Mainland area is outside "
        "the expected sanity range."

    )


print(
    "✓ Mainland boundary "
    "passes sanity check."
)


# ============================================================
# CONVERT TO WGS84 FOR WORLDPOP
# ============================================================

mainland_wgs84 = (

    gpd.GeoSeries(

        [mainland_union],

        crs=METRIC_CRS

    )

    .to_crs(WGS84_CRS)

    .iloc[0]

)


# ============================================================
# 2F. LOAD / DOWNLOAD WORLDPOP 2023
# ============================================================

def download_file(
    url,
    output_path
):

    session = requests.Session()

    retry_strategy = Retry(

        total=3,

        connect=3,

        read=3,

        backoff_factor=2,

        status_forcelist=[
            429,
            500,
            502,
            503,
            504
        ],

        allowed_methods=[
            "GET"
        ]

    )

    adapter = HTTPAdapter(
        max_retries=retry_strategy
    )

    session.mount(
        "https://",
        adapter
    )


    with session.get(
        url,
        stream=True,
        timeout=(30, 300)
    ) as response:

        response.raise_for_status()

        with open(
            output_path,
            "wb"
        ) as file:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if chunk:

                    file.write(chunk)


if os.path.exists(
    WORLDPOP_PATH
):

    file_size_mb = (
        os.path.getsize(
            WORLDPOP_PATH
        )
        / 1_000_000
    )

    print(
        "\nWorldPop raster already exists."
    )

    print(
        f"Using local file: "
        f"{WORLDPOP_PATH}"
    )

    print(
        f"File size: "
        f"{file_size_mb:,.2f} MB"
    )

else:

    print(
        "\nWorldPop raster not found."
    )

    print(
        "Downloading WorldPop 2023 raster..."
    )

    download_file(
        WORLDPOP_URL,
        WORLDPOP_PATH
    )

    print(
        "✓ WorldPop raster downloaded."
    )


# ============================================================
# 2G. POPULATION SUM FUNCTION
# ============================================================

def population_sum(
    raster_path,
    geometry_wgs84
):

    with rasterio.open(
        raster_path
    ) as src:

        geometry = (
            gpd.GeoSeries(
                [geometry_wgs84],
                crs=WGS84_CRS
            )
            .to_crs(src.crs)
            .iloc[0]
        )

        out, _ = mask(
            src,
            [mapping(geometry)],
            crop=True,
            filled=False,
            all_touched=False
        )

        values = (
            out[0]
            .compressed()
            .astype(float)
        )

        values = values[
            np.isfinite(values)
        ]

        values = values[
            values >= 0
        ]

        return float(
            values.sum()
        )


# ============================================================
# 2H. TOTAL MAINLAND POPULATION
# ============================================================

total_mainland_population_2023 = (
    population_sum(
        WORLDPOP_PATH,
        mainland_wgs84
    )
)


if (
    total_mainland_population_2023 <= 0
):

    raise ValueError(
        "Invalid mainland population total."
    )


print("\n=== WORLDPOP 2023 ===")

print(
    "Mainland Jakarta population:",
    f"{total_mainland_population_2023:,.2f}"
)


# ============================================================
# 2I. ACCESSIBILITY GEOMETRY
# ============================================================

def create_access_geometry(
    points_wgs84,
    radius_m
):

    points_metric = (
        points_wgs84
        .to_crs(METRIC_CRS)
    )


    buffers = (
        points_metric
        .geometry
        .buffer(radius_m)
    )


    # Union prevents double counting
    # where multiple facility buffers overlap.

    union_buffers = unary_union(
        buffers
    )


    # Restrict accessibility area
    # to mainland Jakarta.

    clipped = (
        union_buffers
        .intersection(
            mainland_union
        )
    )


    if clipped.is_empty:

        raise ValueError(
            f"Empty accessibility geometry "
            f"for radius {radius_m} m."
        )


    return (
        gpd.GeoSeries(
            [clipped],
            crs=METRIC_CRS
        )
        .to_crs(WGS84_CRS)
        .iloc[0]
    )


# ============================================================
# 2J. CALCULATE C2
# ============================================================

c2_rows = []


for radius_m in SENSITIVITY_RADII_M:

    for mode, points in mode_points.items():

        access_geometry = (
            create_access_geometry(
                points,
                radius_m
            )
        )


        population_covered = (
            population_sum(
                WORLDPOP_PATH,
                access_geometry
            )
        )


        coverage = (
            population_covered
            /
            total_mainland_population_2023
        )


        c2_rows.append({

            "mode": mode,

            "access_radius_m": radius_m,

            "facility_point_count": len(points),

            "population_covered_2023":
                population_covered,

            "total_mainland_population_2023":
                total_mainland_population_2023,

            "population_access_coverage":
                coverage,

            "population_access_coverage_pct":
                coverage * 100

        })

c2_accessibility = pd.DataFrame(
    c2_rows
)


# ============================================================
# 2K. C2 VALIDATION
# ============================================================

if (
    c2_accessibility[
        "population_access_coverage"
    ]
    .isna()
    .any()
):

    raise ValueError(
        "C2 contains missing values."
    )


if (
    ~c2_accessibility[
        "population_access_coverage"
    ]
    .between(0, 1)
    .all()
):

    raise ValueError(
        "C2 contains values outside [0, 1]."
    )


if (
    c2_accessibility[
        "population_covered_2023"
    ]
    >
    c2_accessibility[
        "total_mainland_population_2023"
    ]
).any():

    raise ValueError(
        "Covered population cannot exceed "
        "total mainland population."
    )


print(
    "\n=== C2 — POPULATION ACCESS COVERAGE ==="
)

print(
    c2_accessibility
    .sort_values(
        ["access_radius_m", "mode"]
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 2L. PRIMARY C2 = 500 METERS
# ============================================================

c2_primary = (
    c2_accessibility[
        c2_accessibility[
            "access_radius_m"
        ] == PRIMARY_RADIUS_M
    ]
    .copy()
)


if set(
    c2_primary["mode"]
) != set(
    EXPECTED_COUNTS.keys()
):

    raise ValueError(
        "Primary C2 does not contain all three modes."
    )


# ============================================================
# 2M. SAVE C2 OUTPUT
# ============================================================

c2_accessibility.to_csv(
    "c2_population_access_sensitivity_2026.csv",
    index=False
)

c2_primary.to_csv(
    "c2_population_access_coverage_500m_2026.csv",
    index=False
)


print("\nCreated:")

print(
    "- c2_population_access_sensitivity_2026.csv"
)

print(
    "- c2_population_access_coverage_500m_2026.csv"
)


# ============================================================
# 2N. LINK C2 TO 729-ROW MASTER DATASET
# ============================================================

c2_lookup = (
    c2_primary[
        [
            "mode",
            "facility_point_count",
            "population_covered_2023",
            "total_mainland_population_2023",
            "population_access_coverage",
            "population_access_coverage_pct"
        ]
    ]
    .copy()
)


c2_lookup = c2_lookup.rename(
    columns={
        "facility_point_count":
            "mapped_point_count"
    }
)


# Remove old C2 columns when rerunning
# inside the same Colab session.

old_c2_columns = [
    "mapped_point_count",
    "spatial_network_coverage",
    "spatial_network_coverage_pct",
    "population_covered_2023",
    "total_mainland_population_2023",
    "population_access_coverage",
    "population_access_coverage_pct"
]

df_master = df_master.drop(
    columns=old_c2_columns,
    errors="ignore"
)

df_master = df_master.merge(
    c2_lookup,
    on="mode",
    how="left",
    validate="many_to_one"
)


print("\n=== C2 LINK VALIDATION ===")

print(
    "Rows:",
    len(df_master)
)

print(
    "Missing population_access_coverage:",
    int(
        df_master[
            "population_access_coverage"
        ]
        .isna()
        .sum()
    )
)


if (
    df_master[
        "population_access_coverage"
    ]
    .isna()
    .any()
):

    raise ValueError(
        "C2 failed to link to every observation."
    )


print(
    "✓ C2 successfully linked "
    "to all observations."
)


# ============================================================
# 4. LOAD LAYER C — UMP DKI JAKARTA 2026
# ============================================================

df_ump = pd.read_csv(
    'Filedata Data Upah Minimum di Provinsi DKI Jakarta.csv'
)

TARGET_YEAR = 2026

row_target = df_ump[
    df_ump.astype(str)
    .apply(
        lambda row: row.str.contains(
            str(TARGET_YEAR),
            regex=False
        ).any(),
        axis=1
    )
]

try:

    numeric_values = pd.to_numeric(
        row_target.values.flatten(),
        errors='coerce'
    )

    numeric_values = pd.Series(
        numeric_values
    ).dropna()

    numeric_values = numeric_values[
        numeric_values > 1_000_000
    ]

    ump_2026 = numeric_values.iloc[0]

except (IndexError, ValueError):

    ump_2026 = 5_729_876


df_master['ump_2026'] = ump_2026

print(
    f"\nUMP DKI Jakarta 2026: Rp{ump_2026:,.0f}"
)


# ============================================================
# 5. REFERENCE FARE
# ============================================================

fares = {
    'transjakarta': 3_500,
    'mrt': 14_000,
    'lrt': 5_000
}

df_master['reference_fare'] = (
    df_master['mode'].map(fares)
)


# ============================================================
# 6. C3 — FARE AFFORDABILITY BURDEN
# ============================================================

STANDARD_MONTHLY_TRIPS = 60

df_master['fare_affordability_burden'] = (
    df_master['reference_fare']
    * STANDARD_MONTHLY_TRIPS
    / df_master['ump_2026']
)


# ============================================================
# 7. BUILD FINAL CRITERION-STATE DATASET
# ============================================================

final_cols = [
    'date',
    'day_of_week',
    'is_weekend',
    'mode',

    # C1
    'passenger_count',

    # Spatial supporting variable
    'mapped_point_count',

    # C2
    'population_covered_2023',
    'total_mainland_population_2023',
    'population_access_coverage',
    'population_access_coverage_pct',

    # Economic benchmark
    'ump_2026',

    # C3
    'fare_affordability_burden',

    # Baseline/reference fare
    'reference_fare'
]

df_final = (
    df_master[final_cols]
    .sort_values(
        by=['date', 'mode']
    )
    .reset_index(drop=True)
)


# ============================================================
# 8. VALIDATE CRITERION-STATE DATASET
# ============================================================

print("\n=== DATASET VALIDATION ===")

print(
    f"Total rows: {len(df_final)}"
)

print(
    f"Date range: "
    f"{df_final['date'].min().date()} "
    f"to "
    f"{df_final['date'].max().date()}"
)

print(
    f"Unique dates: "
    f"{df_final['date'].nunique()}"
)

print(
    f"Unique modes: "
    f"{df_final['mode'].nunique()}"
)

print(
    f"Expected rows: "
    f"{243 * 3}"
)

print("\nMode distribution:")

print(
    df_final['mode']
    .value_counts()
)

print("\nMissing values:")

print(
    df_final.isna().sum()
)


# ============================================================
# 9. ENSURE NUMERIC DATA TYPES
# ============================================================

numeric_cols = [
    'passenger_count',
    'mapped_point_count',
    'population_covered_2023',
    'total_mainland_population_2023',
    'population_access_coverage',
    'population_access_coverage_pct',
    'ump_2026',
    'fare_affordability_burden',
    'reference_fare'
]

for col in numeric_cols:

    df_final[col] = pd.to_numeric(
        df_final[col],
        errors='coerce'
    )


# ------------------------------------------------------------
# Validate numeric data
# ------------------------------------------------------------

missing_numeric = (
    df_final[numeric_cols]
    .isna()
    .sum()
)

print("\n=== NUMERIC VALIDATION ===")

print(
    df_final[numeric_cols].dtypes
)

print("\nMissing values:")

print(missing_numeric)

if missing_numeric.sum() > 0:

    raise ValueError(
        "Raw criterion dataset masih memiliki missing values."
    )

print(
    "\n✓ All raw criterion values are valid."
)


# ============================================================
# 10. SAVE CRITERION-STATE DATASET
# ============================================================

criterion_output = (
    'criterion_state_dataset_2026.csv'
)

df_final.to_csv(
    criterion_output,
    index=False
)

print(
    f"\nFile successfully created: "
    f"{criterion_output}"
)


# ============================================================
# 11. MCDM NORMALIZATION
# ============================================================

df_mcdm = df_final.copy()


# ------------------------------------------------------------
# C1 — Relative Demand Intensity
# Benefit criterion
#
# Normalize demand relative to the observed peak of each mode,
# rather than the peak across all modes.
#
# This prevents large-scale modes from dominating the MCDM
# merely because their absolute ridership is larger.
# ------------------------------------------------------------

mode_peak_demand = (
    df_mcdm
    .groupby('mode')['passenger_count']
    .transform('max')
)

if (
    mode_peak_demand <= 0
).any():

    raise ValueError(
        "At least one mode has an invalid peak demand."
    )

df_mcdm['C1_demand_normalized'] = (
    df_mcdm['passenger_count']
    / mode_peak_demand
)


# ------------------------------------------------------------
# C2 — Population Access Coverage
# Benefit criterion
# ------------------------------------------------------------

max_population_access_coverage = (
    df_mcdm[
        'population_access_coverage'
    ]
    .max()
)


if max_population_access_coverage <= 0:

    raise ValueError(
        'Invalid maximum population access coverage.'
    )


df_mcdm['C2_coverage_normalized'] = (

    df_mcdm[
        'population_access_coverage'
    ]

    / max_population_access_coverage

)


# ------------------------------------------------------------
# C3 — Fare Affordability Burden
# Cost criterion
# ------------------------------------------------------------

min_affordability_burden = (
    df_mcdm['fare_affordability_burden'].min()
)

df_mcdm['C3_affordability_normalized'] = (
    min_affordability_burden
    / df_mcdm['fare_affordability_burden']
)


# ============================================================
# 12. NORMALIZATION VALIDATION
# ============================================================

normalized_cols = [
    'C1_demand_normalized',
    'C2_coverage_normalized',
    'C3_affordability_normalized'
]

print("\n=== NORMALIZED VALUE RANGE ===")

for col in normalized_cols:

    print(
        f"{col}: "
        f"min={df_mcdm[col].min():.6f}, "
        f"max={df_mcdm[col].max():.6f}"
    )


print("\n=== NORMALIZED MISSING VALUES ===")

print(
    df_mcdm[normalized_cols]
    .isna()
    .sum()
)


print("\n=== NORMALIZED OUT-OF-RANGE ===")

for col in normalized_cols:

    outside_range = (
        ~df_mcdm[col].between(0, 1)
    ).sum()

    print(
        f"{col}: {outside_range}"
    )


# ============================================================
# 13. SAVE MCDM DATASET
# ============================================================

mcdm_output = (
    'mcdm_normalized_dataset_2026.csv'
)

df_mcdm.to_csv(
    mcdm_output,
    index=False
)

print(
    f"\nFile successfully created: "
    f"{mcdm_output}"
)

# ============================================================
# 17. ENTROPY WEIGHTING
# ============================================================

entropy_cols = [
    'C1_demand_normalized',
    'C2_coverage_normalized',
    'C3_affordability_normalized'
]

# ------------------------------------------------------------
# 17A. Create normalized decision matrix
# ------------------------------------------------------------

R = df_mcdm[entropy_cols].copy()

print("=== ENTROPY INPUT MATRIX ===")
print(R.head())


# ------------------------------------------------------------
# 17B. Calculate proportion p_ij
# ------------------------------------------------------------

column_sums = R.sum(axis=0)

P = R.div(
    column_sums,
    axis=1
)

print("\n=== ENTROPY PROPORTION MATRIX ===")
print(P.head())


# ------------------------------------------------------------
# 17C. Calculate entropy
# ------------------------------------------------------------

n = len(R)

k = 1 / np.log(n)

# Because all normalized values are > 0 in our dataset,
# log(0) should not occur. Still, use a safe calculation.

P_safe = P.replace(0, np.nan)

entropy = (
    -k
    * (
        P_safe * np.log(P_safe)
    ).sum(axis=0)
)

entropy = entropy.fillna(0)


# ------------------------------------------------------------
# 17D. Degree of diversification
# ------------------------------------------------------------

diversification = 1 - entropy


# ------------------------------------------------------------
# 17E. Calculate objective weights
# ------------------------------------------------------------

weights = (
    diversification
    / diversification.sum()
)


# ============================================================
# 18. ENTROPY RESULT
# ============================================================

entropy_result = pd.DataFrame({
    'criterion': [
        'C1 Demand State',
        'C2 Population Access Coverage',
        'C3 Fare Affordability Burden'
    ],
    'entropy': entropy.values,
    'diversification': diversification.values,
    'weight': weights.values
})


print("\n=== ENTROPY WEIGHTING RESULT ===")

print(
    entropy_result
    .to_string(
        index=False
    )
)

print(
    f"\nTotal weight: "
    f"{weights.sum():.6f}"
)


# ============================================================
# 19. WEIGHT VALIDATION
# ============================================================

if not np.isclose(weights.sum(), 1.0):
    raise ValueError(
        "Entropy weights do not sum to 1."
    )

if (weights < 0).any():
    raise ValueError(
        "Entropy weights contain negative values."
    )

print("\n✓ Entropy weights are valid.")

=== PASSENGER COUNT VALIDATION ===
Missing passenger_count: 0
Data type: int64
=== TRANSPORT GIS POINT VALIDATION ===
transjakarta: 378 points (expected 378)
mrt: 22 points (expected 22)
lrt: 11 points (expected 11)

✓ All transport GIS point counts match the expected dataset.

=== MODE-SPECIFIC GIS POINTS ===
transjakarta: 378 points
mrt: 22 points
lrt: 11 points
✓ Mode-specific transport point sets created.

Loading cached mainland DKI boundary...
✓ Cached boundary validated: 650.40 km²

=== DKI MAINLAND BOUNDARY ===
Mainland area: 650.40 km²
✓ Mainland boundary passes sanity check.

WorldPop raster already exists.
Using local file: idn_pop_2023_CN_100m_R2025A_v1.tif
File size: 168.73 MB

=== WORLDPOP 2023 ===
Mainland Jakarta population: 10,886,086.23

=== C2 — POPULATION ACCESS COVERAGE ===
        mode  access_radius_m  facility_point_count  population_covered_2023  total_mainland_population_2023  population_access_coverage  population_access_coverage_pct
         lrt             

In [ ]:


# ============================================================
# 20. SAW WEIGHTED SCORE
# ============================================================

# ------------------------------------------------------------
# Map entropy weights to normalized criteria
# ------------------------------------------------------------

weight_map = {
    'C1_demand_normalized': weights.iloc[0],
    'C2_coverage_normalized': weights.iloc[1],
    'C3_affordability_normalized': weights.iloc[2]
}

print("=== SAW WEIGHTS ===")

for criterion, weight in weight_map.items():
    print(f"{criterion}: {weight:.6f}")


# ------------------------------------------------------------
# Calculate SAW score
# ------------------------------------------------------------

df_mcdm['saw_score'] = (
    df_mcdm['C1_demand_normalized']
    * weight_map['C1_demand_normalized']
    +
    df_mcdm['C2_coverage_normalized']
    * weight_map['C2_coverage_normalized']
    +
    df_mcdm['C3_affordability_normalized']
    * weight_map['C3_affordability_normalized']
)


# ============================================================
# 21. SAW SCORE VALIDATION
# ============================================================

print("\n=== SAW SCORE VALIDATION ===")

print(
    f"Min SAW score: "
    f"{df_mcdm['saw_score'].min():.6f}"
)

print(
    f"Max SAW score: "
    f"{df_mcdm['saw_score'].max():.6f}"
)

print(
    f"Missing SAW score: "
    f"{df_mcdm['saw_score'].isna().sum()}"
)


# Since all normalized criteria and weights are within [0,1],
# the resulting SAW score should also be within [0,1].

outside_saw_range = (
    ~df_mcdm['saw_score'].between(0, 1)
).sum()

print(
    f"SAW score outside [0,1]: "
    f"{outside_saw_range}"
)


# ============================================================
# 22. SHOW SAMPLE SAW RESULTS
# ============================================================

saw_cols = [
    'date',
    'mode',
    'C1_demand_normalized',
    'C2_coverage_normalized',
    'C3_affordability_normalized',
    'saw_score'
]

print("\n=== SAMPLE SAW RESULTS ===")

print(
    df_mcdm[saw_cols]
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 23. DAILY MODE RANKING
# ============================================================

df_mcdm['daily_rank'] = (
    df_mcdm
    .groupby('date')['saw_score']
    .rank(
        ascending=False,
        method='min'
    )
)

print("\n=== SAMPLE DAILY RANKING ===")

print(
    df_mcdm[
        [
            'date',
            'mode',
            'saw_score',
            'daily_rank'
        ]
    ]
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 24. MODE-LEVEL SAW SUMMARY
# ============================================================

saw_summary = (
    df_mcdm
    .groupby('mode')
    .agg(
        average_saw_score=('saw_score', 'mean'),
        min_saw_score=('saw_score', 'min'),
        max_saw_score=('saw_score', 'max'),
        average_daily_rank=('daily_rank', 'mean')
    )
    .reset_index()
)

print("\n=== MODE-LEVEL SAW SUMMARY ===")

print(
    saw_summary
    .to_string(index=False)
)


# ============================================================
# 25. SAVE SAW DATASET
# ============================================================

saw_output = 'mcdm_saw_results_2026.csv'

df_mcdm.to_csv(
    saw_output,
    index=False
)

print(
    f"\nFile successfully created: "
    f"{saw_output}"
)

# ============================================================
# 26. MODE-RELATIVE CONDITION INDEX (RCI)
# ============================================================

# SAW score represents the composite transportation condition
# of each date-mode observation.
#
# To make scores comparable within each mode, calculate the
# historical mean SAW score of each mode as its own baseline.

mode_baseline = (
    df_mcdm
    .groupby('mode')['saw_score']
    .mean()
    .rename('mode_baseline_saw')
)

# Attach mode baseline to each observation
df_mcdm = df_mcdm.merge(
    mode_baseline,
    on='mode',
    how='left'
)

# Relative Condition Index
#
# RCI = current SAW / mode baseline SAW
#
# Interpretation:
# RCI = 1   -> exactly at mode's historical baseline
# RCI > 1   -> above mode's historical baseline
# RCI < 1   -> below mode's historical baseline

df_mcdm['relative_condition_index'] = (
    df_mcdm['saw_score']
    / df_mcdm['mode_baseline_saw']
)


# ============================================================
# 27. VALIDATE RELATIVE CONDITION INDEX
# ============================================================

print("\n=== MODE BASELINE SAW ===")

print(
    mode_baseline
    .to_string()
)

print("\n=== RELATIVE CONDITION SAMPLE ===")

print(
    df_mcdm[
        [
            'date',
            'mode',
            'saw_score',
            'mode_baseline_saw',
            'relative_condition_index'
        ]
    ]
    .head(15)
    .to_string(index=False)
)

print("\n=== RELATIVE CONDITION RANGE ===")

relative_condition_summary = (
    df_mcdm[
        [
            'mode',
            'relative_condition_index'
        ]
    ]
    .groupby('mode')
    .agg(
        min_index=(
            'relative_condition_index',
            'min'
        ),
        mean_index=(
            'relative_condition_index',
            'mean'
        ),
        max_index=(
            'relative_condition_index',
            'max'
        )
    )
    .reset_index()
)

print(
    relative_condition_summary
    .to_string(index=False)
)

# Mean RCI for each mode should be approximately 1
rci_mean_check = (
    df_mcdm
    .groupby('mode')['relative_condition_index']
    .mean()
)

if not np.allclose(
    rci_mean_check.values,
    1.0,
    atol=1e-10
):
    raise ValueError(
        "RCI mean per mode should be approximately 1."
    )

print(
    "\n✓ Relative Condition Index validated."
)


# ============================================================
# 28. FARE ADJUSTMENT SENSITIVITY ANALYSIS
# ============================================================

# Alpha is the sensitivity parameter of the pricing rule.
#
# IMPORTANT:
# Alpha is NOT obtained from Entropy.
# Alpha is a model/policy parameter that controls how strongly
# the fare reacts to the deviation from the mode baseline.

alpha_values = [
    0.05,
    0.10,
    0.15
]

sensitivity_results = []

for alpha in alpha_values:

    temp = df_mcdm.copy()

    # --------------------------------------------------------
    # Relative deviation from mode baseline
    # --------------------------------------------------------

    temp['relative_deviation'] = (
        temp['relative_condition_index']
        - 1
    )

    # --------------------------------------------------------
    # Pricing adjustment percentage
    #
    # RCI < 1 -> negative adjustment
    # RCI > 1 -> positive adjustment
    #
    # This is a pricing-rule assumption layered on top of
    # the MCDM result.
    # --------------------------------------------------------

    temp['fare_adjustment_pct'] = (
        alpha
        * temp['relative_deviation']
    )

    # --------------------------------------------------------
    # Raw model-implied fare
    # --------------------------------------------------------

    temp['raw_dynamic_fare'] = (
        temp['reference_fare']
        * (
            1
            + temp['fare_adjustment_pct']
        )
    )

    # --------------------------------------------------------
    # Summary by mode
    # --------------------------------------------------------

    summary = (
        temp
        .groupby('mode')
        .agg(
            min_condition=(
                'relative_condition_index',
                'min'
            ),
            max_condition=(
                'relative_condition_index',
                'max'
            ),
            min_adjustment_pct=(
                'fare_adjustment_pct',
                'min'
            ),
            max_adjustment_pct=(
                'fare_adjustment_pct',
                'max'
            ),
            min_raw_fare=(
                'raw_dynamic_fare',
                'min'
            ),
            max_raw_fare=(
                'raw_dynamic_fare',
                'max'
            )
        )
        .reset_index()
    )

    summary['alpha'] = alpha

    sensitivity_results.append(summary)


# Combine sensitivity scenarios
sensitivity_df = pd.concat(
    sensitivity_results,
    ignore_index=True
)

print("\n=== FARE ADJUSTMENT SENSITIVITY ===")

print(
    sensitivity_df[
        [
            'alpha',
            'mode',
            'min_adjustment_pct',
            'max_adjustment_pct',
            'min_raw_fare',
            'max_raw_fare'
        ]
    ]
    .to_string(index=False)
)

# Save sensitivity results
sensitivity_df.to_csv(
    'fare_adjustment_sensitivity_2026.csv',
    index=False
)


# ============================================================
# 29. RAW DYNAMIC FARE RECOMMENDATION
# ============================================================

# Primary scenario
ALPHA = 0.10

# Relative deviation from mode baseline
df_mcdm['relative_deviation'] = (
    df_mcdm['relative_condition_index']
    - 1
)

# Percentage adjustment
df_mcdm['fare_adjustment_pct'] = (
    ALPHA
    * df_mcdm['relative_deviation']
)

# Amount of adjustment in Rupiah
df_mcdm['raw_adjustment_amount'] = (
    df_mcdm['reference_fare']
    * df_mcdm['fare_adjustment_pct']
)

# Raw model-implied fare
#
# IMPORTANT:
# This is the output of the MCDM + pricing rule BEFORE
# applying current policy constraints.

df_mcdm['raw_dynamic_fare'] = (
    df_mcdm['reference_fare']
    + df_mcdm['raw_adjustment_amount']
)


# ============================================================
# 30. RAW FARE VALIDATION
# ============================================================

if (
    df_mcdm['raw_dynamic_fare']
    .isna()
    .any()
):
    raise ValueError(
        "Raw dynamic fare contains missing values."
    )

if (
    df_mcdm['raw_dynamic_fare']
    <= 0
).any():
    raise ValueError(
        "Raw dynamic fare contains zero or negative values."
    )

print("\n=== RAW FARE VALIDATION ===")

print(
    f"Minimum raw fare: "
    f"Rp{df_mcdm['raw_dynamic_fare'].min():,.2f}"
)

print(
    f"Maximum raw fare: "
    f"Rp{df_mcdm['raw_dynamic_fare'].max():,.2f}"
)

print(
    f"Missing raw fare: "
    f"{df_mcdm['raw_dynamic_fare'].isna().sum()}"
)

print(
    "\n✓ Raw dynamic fare is valid."
)


# ============================================================
# 31. CURRENT POLICY CONSTRAINT
# ============================================================

# Policy boundaries used in the research simulation.
#
# TransJakarta:
#   fixed/reference fare = Rp3,500
#
# MRT Jakarta:
#   reference benchmark = Rp14,000
#   policy range = Rp3,000–Rp14,000
#
# LRT Jakarta:
#   fixed/reference fare = Rp5,000
#
# These boundaries are treated as constraints on the raw
# research recommendation. They do NOT mean that the resulting
# fare is an actual government-approved dynamic tariff.

policy_min_fare = {
    'transjakarta': 3500,
    'mrt': 3000,
    'lrt': 5000
}

policy_max_fare = {
    'transjakarta': 3500,
    'mrt': 14000,
    'lrt': 5000
}

df_mcdm['policy_min_fare'] = (
    df_mcdm['mode']
    .map(policy_min_fare)
)

df_mcdm['policy_max_fare'] = (
    df_mcdm['mode']
    .map(policy_max_fare)
)


# ------------------------------------------------------------
# Apply policy clipping
# ------------------------------------------------------------

df_mcdm['policy_constrained_fare'] = (
    df_mcdm['raw_dynamic_fare']
    .clip(
        lower=df_mcdm['policy_min_fare'],
        upper=df_mcdm['policy_max_fare']
    )
)


# ============================================================
# 32. POLICY CONSTRAINT ANALYSIS
# ============================================================

# Amount of raw change proposed by the model
df_mcdm['raw_fare_adjustment_amount'] = (
    df_mcdm['raw_dynamic_fare']
    - df_mcdm['reference_fare']
)

# Amount of change remaining after policy constraint
df_mcdm['final_fare_adjustment_amount'] = (
    df_mcdm['policy_constrained_fare']
    - df_mcdm['reference_fare']
)

# Percentage change remaining after policy constraint
df_mcdm['final_fare_adjustment_pct'] = (
    df_mcdm['final_fare_adjustment_amount']
    / df_mcdm['reference_fare']
)


# ------------------------------------------------------------
# Determine whether policy constraint changed the raw result
# ------------------------------------------------------------

df_mcdm['policy_constraint_applied'] = (
    ~np.isclose(
        df_mcdm['raw_dynamic_fare'],
        df_mcdm['policy_constrained_fare'],
        rtol=0,
        atol=1e-9
    )
)


# ------------------------------------------------------------
# Direction of raw model pressure
# ------------------------------------------------------------

df_mcdm['raw_pricing_direction'] = np.select(
    [
        df_mcdm['raw_fare_adjustment_amount'] > 0,
        df_mcdm['raw_fare_adjustment_amount'] < 0
    ],
    [
        'increase',
        'decrease'
    ],
    default='no_change'
)


# ------------------------------------------------------------
# Direction of final feasible recommendation
# ------------------------------------------------------------

df_mcdm['final_pricing_direction'] = np.select(
    [
        df_mcdm['final_fare_adjustment_amount'] > 0,
        df_mcdm['final_fare_adjustment_amount'] < 0
    ],
    [
        'increase',
        'decrease'
    ],
    default='no_change'
)


# ============================================================
# 33. RAW UPWARD / DOWNWARD PRICING PRESSURE
# ============================================================

# These are NOT actual subsidies or government funding needs.
#
# They simply describe how much the raw model recommendation
# is above or below the reference fare.

df_mcdm['model_implied_upward_pressure'] = (
    df_mcdm['raw_dynamic_fare']
    - df_mcdm['reference_fare']
).clip(lower=0)

df_mcdm['model_implied_downward_pressure'] = (
    df_mcdm['reference_fare']
    - df_mcdm['raw_dynamic_fare']
).clip(lower=0)


# ============================================================
# 34. FINAL RECOMMENDED FARE
# ============================================================

# The final fare used by the research output is the
# policy-constrained fare.
#
# This is intentionally different from the raw model-implied
# fare:
#
# raw_dynamic_fare
#     = mathematical recommendation before policy constraints
#
# policy_constrained_fare
#     = recommendation after applying policy boundaries
#
# final_recommended_fare
#     = policy-constrained research recommendation

df_mcdm['final_recommended_fare'] = (
    df_mcdm['policy_constrained_fare']
)


# ============================================================
# 35. FINAL AFFORDABILITY
# ============================================================

STANDARD_MONTHLY_TRIPS = 60

# Baseline monthly transport cost
df_mcdm['baseline_monthly_transport_cost'] = (
    df_mcdm['reference_fare']
    * STANDARD_MONTHLY_TRIPS
)

# Final monthly transport cost
df_mcdm['final_monthly_transport_cost'] = (
    df_mcdm['final_recommended_fare']
    * STANDARD_MONTHLY_TRIPS
)

# Baseline affordability burden
df_mcdm['baseline_affordability_burden'] = (
    df_mcdm['baseline_monthly_transport_cost']
    / df_mcdm['ump_2026']
)

# Final affordability burden
df_mcdm['final_affordability_burden'] = (
    df_mcdm['final_monthly_transport_cost']
    / df_mcdm['ump_2026']
)


# ============================================================
# 36. FINAL VALIDATION
# ============================================================

print("\n=== FINAL FARE VALIDATION ===")

# ------------------------------------------------------------
# 36A. Policy lower-bound violations
# ------------------------------------------------------------

lower_bound_violations = (
    df_mcdm['final_recommended_fare']
    < df_mcdm['policy_min_fare']
).sum()

# ------------------------------------------------------------
# 36B. Policy upper-bound violations
# ------------------------------------------------------------

upper_bound_violations = (
    df_mcdm['final_recommended_fare']
    > df_mcdm['policy_max_fare']
).sum()

# ------------------------------------------------------------
# 36C. Negative fares
# ------------------------------------------------------------

negative_fares = (
    df_mcdm['final_recommended_fare']
    <= 0
).sum()

# ------------------------------------------------------------
# 36D. Missing values
# ------------------------------------------------------------

missing_final_fare = (
    df_mcdm['final_recommended_fare']
    .isna()
    .sum()
)

print(
    f"Lower-bound violations: "
    f"{lower_bound_violations}"
)

print(
    f"Upper-bound violations: "
    f"{upper_bound_violations}"
)

print(
    f"Zero/negative final fares: "
    f"{negative_fares}"
)

print(
    f"Missing final fares: "
    f"{missing_final_fare}"
)


if lower_bound_violations > 0:
    raise ValueError(
        "Final fares violate policy minimum."
    )

if upper_bound_violations > 0:
    raise ValueError(
        "Final fares violate policy maximum."
    )

if negative_fares > 0:
    raise ValueError(
        "Final fares contain zero or negative values."
    )

if missing_final_fare > 0:
    raise ValueError(
        "Final fares contain missing values."
    )


print(
    "\n✓ Final policy-constrained fares are valid."
)


# ------------------------------------------------------------
# 36E. Affordability validation
#
# Final fare can not exceed the reference fare in the current
# policy/reference configuration:
#   TJ  = 3500 fixed
#   LRT = 5000 fixed
#   MRT = 14000 reference = policy maximum
#
# Therefore the final affordability burden should not exceed
# its baseline value.
# ------------------------------------------------------------

affordability_violations = (
    df_mcdm['final_affordability_burden']
    > df_mcdm['baseline_affordability_burden']
    + 1e-12
).sum()

print(
    f"Affordability violations: "
    f"{affordability_violations}"
)

if affordability_violations > 0:
    raise ValueError(
        "Final affordability burden exceeds baseline."
    )

print(
    "\n✓ Final affordability constraint validated."
)


# ============================================================
# 37. SAMPLE FINAL RESULTS
# ============================================================

fare_cols = [
    'date',
    'mode',

    'reference_fare',

    'saw_score',
    'relative_condition_index',

    'fare_adjustment_pct',

    'raw_dynamic_fare',

    'policy_min_fare',
    'policy_max_fare',

    'policy_constrained_fare',

    'final_recommended_fare',

    'policy_constraint_applied',

    'raw_pricing_direction',
    'final_pricing_direction',

    'raw_fare_adjustment_amount',
    'final_fare_adjustment_amount',

    'model_implied_upward_pressure',
    'model_implied_downward_pressure',

    'baseline_affordability_burden',
    'final_affordability_burden'
]

print("\n=== SAMPLE FINAL FARE RESULTS ===")

print(
    df_mcdm[
        fare_cols
    ]
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 38. FARE SUMMARY BY MODE
# ============================================================

print("\n=== FINAL FARE RANGE BY MODE ===")

final_fare_summary = (
    df_mcdm
    .groupby('mode')
    .agg(
        min_raw_fare=(
            'raw_dynamic_fare',
            'min'
        ),

        max_raw_fare=(
            'raw_dynamic_fare',
            'max'
        ),

        min_final_fare=(
            'final_recommended_fare',
            'min'
        ),

        max_final_fare=(
            'final_recommended_fare',
            'max'
        ),

        mean_final_fare=(
            'final_recommended_fare',
            'mean'
        ),

        mean_final_adjustment_pct=(
            'final_fare_adjustment_pct',
            'mean'
        ),

        max_upward_pressure=(
            'model_implied_upward_pressure',
            'max'
        ),

        max_downward_pressure=(
            'model_implied_downward_pressure',
            'max'
        ),

        policy_constraints_applied=(
            'policy_constraint_applied',
            'sum'
        )
    )
    .reset_index()
)

print(
    final_fare_summary
    .to_string(index=False)
)


# ============================================================
# 39. POLICY CONSTRAINT SUMMARY
# ============================================================

policy_constraint_summary = (
    df_mcdm
    .groupby('mode')
    .agg(
        total_observations=(
            'policy_constraint_applied',
            'count'
        ),
        constrained_observations=(
            'policy_constraint_applied',
            'sum'
        )
    )
    .reset_index()
)

policy_constraint_summary['constraint_percentage'] = (
    policy_constraint_summary['constrained_observations']
    / policy_constraint_summary['total_observations']
    * 100
)

print("\n=== POLICY CONSTRAINT SUMMARY ===")

print(
    policy_constraint_summary
    .to_string(index=False)
)


# ============================================================
# 40. SAVE ENTROPY RESULTS
# ============================================================

entropy_result.to_csv(
    'entropy_weight_results_2026.csv',
    index=False
)

print(
    "\nEntropy weight file created: "
    "entropy_weight_results_2026.csv"
)


# ============================================================
# 41. SAVE SAW RESULTS
# ============================================================

saw_output = (
    'mcdm_saw_results_2026.csv'
)

df_mcdm.to_csv(
    saw_output,
    index=False
)

print(
    "\nSAW dataset successfully created: "
    f"{saw_output}"
)


# ============================================================
# 42. SAVE FINAL DYNAMIC FARE DATASET
# ============================================================

final_output = (
    'dynamic_fare_results_2026.csv'
)

df_mcdm.to_csv(
    final_output,
    index=False
)

print(
    "\nFinal dataset successfully created: "
    f"{final_output}"
)


=== SAW WEIGHTS ===
C1_demand_normalized: 0.051820
C2_coverage_normalized: 0.799802
C3_affordability_normalized: 0.148377

=== SAW SCORE VALIDATION ===
Min SAW score: 0.090121
Max SAW score: 1.000000
Missing SAW score: 0
SAW score outside [0,1]: 0

=== SAMPLE SAW RESULTS ===
      date         mode  C1_demand_normalized  C2_coverage_normalized  C3_affordability_normalized  saw_score
2023-01-01          lrt              0.410785                0.068795                         0.70   0.180173
2023-01-01          mrt              0.330837                0.055966                         0.25   0.099000
2023-01-01 transjakarta              0.247875                1.000000                         1.00   0.961025
2023-01-02          lrt              0.554175                0.068795                         0.70   0.187604
2023-01-02          mrt              0.464483                0.055966                         0.25   0.105926
2023-01-02 transjakarta              0.567906                1.0

In [ ]:
# ============================================================
# 43. MODEL VALIDATION & ROBUSTNESS ANALYSIS
# ============================================================

print("\n")
print("=" * 70)
print("MODEL VALIDATION & ROBUSTNESS ANALYSIS")
print("=" * 70)


# ============================================================
# 43A. DEFINE BASELINE MODEL COMPONENTS
# ============================================================

criterion_cols = [
    'C1_demand_normalized',
    'C2_coverage_normalized',
    'C3_affordability_normalized'
]

baseline_weights = pd.Series(
    {
        'C1_demand_normalized': weights.iloc[0],
        'C2_coverage_normalized': weights.iloc[1],
        'C3_affordability_normalized': weights.iloc[2]
    }
)

print("\n=== BASELINE WEIGHTS ===")

print(
    baseline_weights.to_string()
)

print(
    f"\nWeight sum: "
    f"{baseline_weights.sum():.6f}"
)




# ============================================================
# 43B. NORMALIZATION FUNCTIONS
# ============================================================

def normalize_criteria(data):
    """
    Normalize the three MCDM criteria.

    C1:
        Relative Demand Intensity
        Benefit criterion
        Normalized against the observed peak demand
        within each transportation mode.

    C2:
        Population Access Coverage
        Benefit criterion
        Uses population_access_coverage.

    C3:
        Fare Affordability Burden
        Cost criterion.
    """

    data = data.copy()


    # --------------------------------------------------------
    # C1 — Relative Demand Intensity
    # --------------------------------------------------------

    demand_peak = (
        data
        .groupby('mode')['passenger_count']
        .transform('max')
    )


    if (demand_peak <= 0).any():

        raise ValueError(
            "Demand peak contains zero or negative values."
        )


    data['C1_demand_normalized'] = (
        data['passenger_count']
        /
        demand_peak
    )


    # --------------------------------------------------------
    # C2 — Population Access Coverage
    # --------------------------------------------------------

    c2_max = (
        data['population_access_coverage']
        .max()
    )


    if c2_max <= 0:

        raise ValueError(
            "Maximum population access coverage "
            "must be greater than zero."
        )


    data['C2_coverage_normalized'] = (
        data['population_access_coverage']
        /
        c2_max
    )


    # --------------------------------------------------------
    # C3 — Fare Affordability Burden
    # --------------------------------------------------------

    c3_min = (
        data['fare_affordability_burden']
        .min()
    )


    if c3_min <= 0:

        raise ValueError(
            "Minimum fare affordability burden "
            "must be greater than zero."
        )


    data['C3_affordability_normalized'] = (
        c3_min
        /
        data['fare_affordability_burden']
    )


    # --------------------------------------------------------
    # Validate normalized criteria
    # --------------------------------------------------------

    normalized_cols = [
        'C1_demand_normalized',
        'C2_coverage_normalized',
        'C3_affordability_normalized'
    ]


    if data[
        normalized_cols
    ].isna().any().any():

        raise ValueError(
            "Normalization produced missing values."
        )


    if (
        ~data[
            normalized_cols
        ]
        .apply(
            lambda col: col.between(
                0,
                1
            )
        )
        .all()
        .all()
    ):

        raise ValueError(
            "Normalized criteria contain "
            "values outside [0, 1]."
        )


    return data


# ============================================================
# 43C. REUSABLE ENTROPY FUNCTION
# ============================================================

def calculate_entropy_weights(data):
    """
    Calculate entropy weights from the three normalized
    MCDM criteria.

    The function intentionally selects only criterion_cols,
    so non-numeric columns such as date and mode are never
    included in entropy calculations.
    """


    # --------------------------------------------------------
    # Select ONLY the three MCDM criteria
    # --------------------------------------------------------

    missing_cols = [
        col
        for col in criterion_cols
        if col not in data.columns
    ]


    if missing_cols:

        raise ValueError(
            "Missing entropy criterion columns: "
            f"{missing_cols}"
        )


    R = (
        data[
            criterion_cols
        ]
        .apply(
            pd.to_numeric,
            errors='coerce'
        )
        .copy()
    )


    # --------------------------------------------------------
    # Numeric validation
    # --------------------------------------------------------

    if R.isna().any().any():

        raise ValueError(
            "Entropy input contains "
            "non-numeric or missing values."
        )


    if (
        R < 0
    ).any().any():

        raise ValueError(
            "Entropy input contains negative values."
        )


    # --------------------------------------------------------
    # Column sums
    # --------------------------------------------------------

    column_sums = R.sum(
        axis=0
    )


    if (
        column_sums <= 0
    ).any():

        raise ValueError(
            "At least one entropy criterion "
            "has a zero column sum."
        )


    # --------------------------------------------------------
    # Calculate proportion P_ij
    # --------------------------------------------------------

    P = R.div(
        column_sums,
        axis=1
    )


    # --------------------------------------------------------
    # Safe logarithm
    # --------------------------------------------------------

    P_safe = P.replace(
        0,
        np.nan
    )


    # --------------------------------------------------------
    # Entropy
    # --------------------------------------------------------

    n = len(R)


    if n <= 1:

        raise ValueError(
            "Entropy calculation requires "
            "more than one observation."
        )


    k = 1 / np.log(n)


    entropy_values = (
        -k
        * (
            P_safe
            * np.log(P_safe)
        )
        .sum(
            axis=0
        )
    )


    entropy_values = (
        entropy_values
        .fillna(0)
    )


    # --------------------------------------------------------
    # Diversification
    # --------------------------------------------------------

    diversification_values = (
        1
        - entropy_values
    )


    diversification_sum = (
        diversification_values.sum()
    )


    if diversification_sum <= 0:

        raise ValueError(
            "Total entropy diversification "
            "must be greater than zero."
        )


    # --------------------------------------------------------
    # Objective weights
    # --------------------------------------------------------

    weight_values = (
        diversification_values
        /
        diversification_sum
    )


    return weight_values


# ============================================================
# 44. TEMPORAL WEIGHT STABILITY
# ============================================================

# ------------------------------------------------------------
# Purpose:
#
# Recalculate entropy weights independently for each quarter.
#
# This tests whether the objective criterion weights are
# heavily dependent on a particular time period.
#
# This is NOT predictive validation.
# It is temporal stability analysis.
# ------------------------------------------------------------

df_temporal = df_final.copy()

df_temporal['quarter'] = (
    df_temporal['date']
    .dt.quarter
)

quarter_weight_results = []
quarter_score_results = []

for quarter in sorted(
    df_temporal['quarter'].unique()
):

    quarter_data = (
        df_temporal[
            df_temporal['quarter'] == quarter
        ]
        .copy()
    )

    R_quarter = normalize_criteria(
        quarter_data
    )

    quarter_weights = (
        calculate_entropy_weights(
            R_quarter
        )
    )

    quarter_weights['quarter'] = quarter

    quarter_weight_results.append(
        quarter_weights
    )

    # --------------------------------------------------------
    # Calculate quarter-specific SAW
    # --------------------------------------------------------

    quarter_scores = (
    R_quarter[
        criterion_cols
    ]
    .dot(
        quarter_weights[
            criterion_cols
        ]
      )
    )

    # Compare quarter-specific SAW with baseline SAW
    baseline_scores = (
        df_mcdm
        .loc[
            quarter_data.index,
            'saw_score'
        ]
    )

    rank_correlation = (
        baseline_scores
        .corr(
            quarter_scores,
            method='spearman'
        )
    )

    quarter_score_results.append(
        {
            'quarter': quarter,
            'observations': len(quarter_data),
            'saw_spearman_vs_baseline': (
                rank_correlation
            )
        }
    )


quarter_weights_df = (
    pd.DataFrame(
        quarter_weight_results
    )
    .reset_index(drop=True)
)


quarter_weights_df = (
    quarter_weights_df[
        [
            'quarter',
            'C1_demand_normalized',
            'C2_coverage_normalized',
            'C3_affordability_normalized'
        ]
    ]
)


quarter_score_df = pd.DataFrame(
    quarter_score_results
)


print("\n=== TEMPORAL ENTROPY WEIGHT STABILITY ===")

print(
    quarter_weights_df
    .to_string(index=False)
)

print(
    "\n=== QUARTERLY SAW RANK STABILITY ==="
)

print(
    quarter_score_df
    .to_string(index=False)
)


# ------------------------------------------------------------
# Weight stability statistics
# ------------------------------------------------------------

temporal_weight_summary = pd.DataFrame({
    'criterion': criterion_cols,

    'mean_weight': [
        quarter_weights_df[col].mean()
        for col in criterion_cols
    ],

    'std_weight': [
        quarter_weights_df[col].std(
            ddof=1
        )
        for col in criterion_cols
    ],

    'min_weight': [
        quarter_weights_df[col].min()
        for col in criterion_cols
    ],

    'max_weight': [
        quarter_weights_df[col].max()
        for col in criterion_cols
    ]
})


temporal_weight_summary['range'] = (
    temporal_weight_summary['max_weight']
    - temporal_weight_summary['min_weight']
)


print(
    "\n=== TEMPORAL WEIGHT SUMMARY ==="
)

print(
    temporal_weight_summary
    .to_string(index=False)
)


# Save
quarter_weights_df.to_csv(
    'temporal_entropy_weight_stability_2026.csv',
    index=False
)

quarter_score_df.to_csv(
    'temporal_saw_stability_2026.csv',
    index=False
)

temporal_weight_summary.to_csv(
    'temporal_weight_summary_2026.csv',
    index=False
)


# ============================================================
# 45. BOOTSTRAP WEIGHT STABILITY
# ============================================================

# ------------------------------------------------------------
# Purpose:
#
# Resample observed dates with replacement while keeping the
# three modes belonging to each sampled date together.
#
# This preserves the daily multimodal structure.
#
# The analysis measures how much entropy weights change when
# the observed date sample is perturbed.
# ------------------------------------------------------------

N_BOOTSTRAP = 300

rng = np.random.default_rng(
    42
)

unique_dates = (
    df_final['date']
    .drop_duplicates()
    .sort_values()
    .tolist()
)

date_groups = {
    date: (
        df_final[
            df_final['date'] == date
        ]
        .copy()
    )
    for date in unique_dates
}


bootstrap_weights = []

for iteration in range(
    N_BOOTSTRAP
):

    sampled_date_positions = (
        rng.choice(
            len(unique_dates),
            size=len(unique_dates),
            replace=True
        )
    )

    sampled_parts = [
        date_groups[
            unique_dates[position]
        ]
        for position
        in sampled_date_positions
    ]

    bootstrap_data = pd.concat(
        sampled_parts,
        ignore_index=True
    )

    R_bootstrap = (
        normalize_criteria(
            bootstrap_data
        )
    )

    bootstrap_weight = (
        calculate_entropy_weights(
            R_bootstrap
        )
    )

    bootstrap_weights.append(
        bootstrap_weight.values
    )


bootstrap_weights_df = pd.DataFrame(
    bootstrap_weights,
    columns=criterion_cols
)


# ------------------------------------------------------------
# Bootstrap summary
# ------------------------------------------------------------

bootstrap_summary = pd.DataFrame({
    'criterion': criterion_cols,

    'baseline_weight': [
        baseline_weights[col]
        for col in criterion_cols
    ],

    'bootstrap_mean': [
        bootstrap_weights_df[col].mean()
        for col in criterion_cols
    ],

    'bootstrap_std': [
        bootstrap_weights_df[col].std(
            ddof=1
        )
        for col in criterion_cols
    ],

    'bootstrap_min': [
        bootstrap_weights_df[col].min()
        for col in criterion_cols
    ],

    'bootstrap_max': [
        bootstrap_weights_df[col].max()
        for col in criterion_cols
    ],

    'bootstrap_p2_5': [
        bootstrap_weights_df[col].quantile(
            0.025
        )
        for col in criterion_cols
    ],

    'bootstrap_p97_5': [
        bootstrap_weights_df[col].quantile(
            0.975
        )
        for col in criterion_cols
    ]
})


bootstrap_summary['absolute_mean_difference'] = (
    bootstrap_summary['bootstrap_mean']
    - bootstrap_summary['baseline_weight']
).abs()


bootstrap_summary['relative_mean_difference_pct'] = (
    bootstrap_summary[
        'absolute_mean_difference'
    ]
    / bootstrap_summary['baseline_weight']
    * 100
)


print(
    "\n=== BOOTSTRAP ENTROPY WEIGHT STABILITY ==="
)

print(
    bootstrap_summary
    .to_string(index=False)
)


bootstrap_weights_df.to_csv(
    'bootstrap_entropy_weights_2026.csv',
    index=False
)

bootstrap_summary.to_csv(
    'bootstrap_entropy_weight_summary_2026.csv',
    index=False
)


# ============================================================
# 46. CRITERION WEIGHT PERTURBATION ±20%
# ============================================================

# ------------------------------------------------------------
# Purpose:
#
# Perturb one criterion weight by ±20% and renormalize the
# weight vector.
#
# This answers:
#
# "Would the SAW and fare recommendation materially change
# if the objective weights were slightly different?"
# ------------------------------------------------------------

weight_sensitivity_results = []


baseline_raw_fare = (
    df_mcdm['raw_dynamic_fare']
    .copy()
)


for criterion in criterion_cols:

    for perturbation in [
        -0.20,
        0.20
    ]:

        scenario_weights = (
            baseline_weights
            .copy()
        )

        scenario_weights[
            criterion
        ] *= (
            1 + perturbation
        )

        scenario_weights = (
            scenario_weights
            / scenario_weights.sum()
        )

        scenario_score = (
            df_mcdm[
                criterion_cols
            ]
            .dot(
                scenario_weights[
                    criterion_cols
                ]
            )
        )

        # RCI
        scenario_baseline = (
            pd.DataFrame({
                'mode': df_mcdm['mode'],
                'score': scenario_score
            })
            .groupby('mode')['score']
            .transform('mean')
        )

        scenario_rci = (
            scenario_score
            / scenario_baseline
        )

        # Raw fare
        scenario_raw_fare = (
            df_mcdm['reference_fare']
            * (
                1
                + ALPHA
                * (
                    scenario_rci
                    - 1
                )
            )
        )

        # Policy constraint
        scenario_final_fare = (
            scenario_raw_fare
            .clip(
                lower=df_mcdm[
                    'policy_min_fare'
                ],
                upper=df_mcdm[
                    'policy_max_fare'
                ]
            )
        )

        saw_rank_corr = (
            df_mcdm['saw_score']
            .corr(
                scenario_score,
                method='spearman'
            )
        )

        raw_fare_difference_pct = (
            (
                scenario_raw_fare
                - baseline_raw_fare
            ).abs()
            / df_mcdm[
                'reference_fare'
            ]
            * 100
        )

        final_fare_difference_pct = (
            (
                scenario_final_fare
                - df_mcdm[
                    'final_recommended_fare'
                ]
            ).abs()
            / df_mcdm[
                'reference_fare'
            ]
            * 100
        )

        weight_sensitivity_results.append({
            'criterion_perturbed': criterion,
            'perturbation': perturbation,

            'new_weight': (
                scenario_weights[
                    criterion
                ]
            ),

            'C1_weight': (
                scenario_weights[
                    'C1_demand_normalized'
                ]
            ),

            'C2_weight': (
                scenario_weights[
                    'C2_coverage_normalized'
                ]
            ),

            'C3_weight': (
                scenario_weights[
                    'C3_affordability_normalized'
                ]
            ),

            'SAW_spearman_vs_baseline': (
                saw_rank_corr
            ),

            'mean_absolute_raw_fare_change_pct': (
                raw_fare_difference_pct.mean()
            ),

            'max_absolute_raw_fare_change_pct': (
                raw_fare_difference_pct.max()
            ),

            'mean_absolute_final_fare_change_pct': (
                final_fare_difference_pct.mean()
            ),

            'max_absolute_final_fare_change_pct': (
                final_fare_difference_pct.max()
            )
        })


weight_sensitivity_df = pd.DataFrame(
    weight_sensitivity_results
)


print(
    "\n=== WEIGHT SENSITIVITY ±20% ==="
)

print(
    weight_sensitivity_df
    .to_string(index=False)
)

weight_sensitivity_df.to_csv(
    'weight_sensitivity_2026.csv',
    index=False
)


# ============================================================
# 47. CRITERION EXCLUSION ANALYSIS
# ============================================================

# ------------------------------------------------------------
# Purpose:
#
# Remove one criterion at a time and redistribute its weight
# proportionally to the remaining criteria.
#
# This tests whether the model's conclusions depend completely
# on one particular criterion.
# ------------------------------------------------------------

criterion_exclusion_results = []


for excluded_criterion in criterion_cols:

    retained_criteria = [
        col
        for col in criterion_cols
        if col != excluded_criterion
    ]

    retained_weights = (
        baseline_weights[
            retained_criteria
        ]
        / baseline_weights[
            retained_criteria
        ].sum()
    )

    exclusion_score = (
        df_mcdm[
            retained_criteria
        ]
        .dot(
            retained_weights[
                retained_criteria
            ]
        )
    )

    exclusion_baseline = (
        pd.DataFrame({
            'mode': df_mcdm['mode'],
            'score': exclusion_score
        })
        .groupby('mode')['score']
        .transform('mean')
    )

    exclusion_rci = (
        exclusion_score
        / exclusion_baseline
    )

    exclusion_raw_fare = (
        df_mcdm['reference_fare']
        * (
            1
            + ALPHA
            * (
                exclusion_rci
                - 1
            )
        )
    )

    exclusion_final_fare = (
        exclusion_raw_fare
        .clip(
            lower=df_mcdm[
                'policy_min_fare'
            ],
            upper=df_mcdm[
                'policy_max_fare'
            ]
        )
    )

    saw_rank_corr = (
        df_mcdm['saw_score']
        .corr(
            exclusion_score,
            method='spearman'
        )
    )

    raw_fare_change_pct = (
        (
            exclusion_raw_fare
            - baseline_raw_fare
        ).abs()
        / df_mcdm[
            'reference_fare'
        ]
        * 100
    )

    final_fare_change_pct = (
        (
            exclusion_final_fare
            - df_mcdm[
                'final_recommended_fare'
            ]
        ).abs()
        / df_mcdm[
            'reference_fare'
        ]
        * 100
    )

    criterion_exclusion_results.append({
        'excluded_criterion': (
            excluded_criterion
        ),

        'retained_weight_sum': (
            retained_weights.sum()
        ),

        'SAW_spearman_vs_baseline': (
            saw_rank_corr
        ),

        'mean_absolute_raw_fare_change_pct': (
            raw_fare_change_pct.mean()
        ),

        'max_absolute_raw_fare_change_pct': (
            raw_fare_change_pct.max()
        ),

        'mean_absolute_final_fare_change_pct': (
            final_fare_change_pct.mean()
        ),

        'max_absolute_final_fare_change_pct': (
            final_fare_change_pct.max()
        )
    })


criterion_exclusion_df = pd.DataFrame(
    criterion_exclusion_results
)


print(
    "\n=== CRITERION EXCLUSION ANALYSIS ==="
)

print(
    criterion_exclusion_df
    .to_string(index=False)
)


criterion_exclusion_df.to_csv(
    'criterion_exclusion_robustness_2026.csv',
    index=False
)


# ============================================================
# 48. ALPHA ROBUSTNESS INCLUDING STRESS SCENARIO
# ============================================================

# Existing sensitivity:
#   0.05
#   0.10
#   0.15
#
# Add:
#   0.00 -> static baseline identity check
#   0.20 -> stronger stress scenario

alpha_validation_values = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]

alpha_validation_results = []


for alpha_value in alpha_validation_values:

    validation_raw = (
        df_mcdm['reference_fare']
        * (
            1
            + alpha_value
            * (
                df_mcdm[
                    'relative_condition_index'
                ]
                - 1
            )
        )
    )

    validation_final = (
        validation_raw
        .clip(
            lower=df_mcdm[
                'policy_min_fare'
            ],
            upper=df_mcdm[
                'policy_max_fare'
            ]
        )
    )

    for mode in [
        'lrt',
        'mrt',
        'transjakarta'
    ]:

        mask = (
            df_mcdm['mode'] == mode
        )

        reference = (
            df_mcdm.loc[
                mask,
                'reference_fare'
            ]
        )

        raw = (
            validation_raw[
                mask
            ]
        )

        final = (
            validation_final[
                mask
            ]
        )

        policy_applied = (
            ~np.isclose(
                raw,
                final,
                rtol=0,
                atol=1e-9
            )
        )

        alpha_validation_results.append({
            'alpha': alpha_value,
            'mode': mode,

            'min_raw_fare': raw.min(),
            'max_raw_fare': raw.max(),

            'mean_raw_fare': raw.mean(),

            'mean_absolute_raw_adjustment_pct': (
                (
                    (
                        raw
                        - reference
                    ).abs()
                    / reference
                ).mean()
                * 100
            ),

            'min_final_fare': final.min(),
            'max_final_fare': final.max(),

            'mean_final_fare': final.mean(),

            'mean_absolute_final_adjustment_pct': (
                (
                    (
                        final
                        - reference
                    ).abs()
                    / reference
                ).mean()
                * 100
            ),

            'policy_constraint_rate_pct': (
                policy_applied.mean()
                * 100
            )
        })


alpha_validation_df = pd.DataFrame(
    alpha_validation_results
)


print(
    "\n=== ALPHA ROBUSTNESS ==="
)

print(
    alpha_validation_df
    .to_string(index=False)
)


# ------------------------------------------------------------
# Alpha = 0 identity test
# ------------------------------------------------------------

alpha_zero = (
    alpha_validation_df[
        alpha_validation_df['alpha'] == 0
    ]
)


if not np.allclose(
    alpha_zero['mean_raw_fare'],
    alpha_zero.apply(
        lambda row: df_mcdm[
            df_mcdm['mode']
            == row['mode']
        ]['reference_fare'].mean(),
        axis=1
    )
):

    raise ValueError(
        "Alpha=0 does not reproduce reference fare."
    )


print(
    "\n✓ Alpha=0 correctly reproduces reference fare."
)


alpha_validation_df.to_csv(
    'alpha_robustness_2026.csv',
    index=False
)


# ============================================================
# 49. MODEL MONOTONICITY / SANITY CHECK
# ============================================================

# ------------------------------------------------------------
# Because C2 and C3 are constant within each mode, the
# temporal variation of SAW within a mode is driven by C1.
#
# Therefore, higher demand should never produce a lower SAW
# score, RCI, or raw fare within the same mode.
#
# This is an implementation consistency test.
# ------------------------------------------------------------

monotonicity_results = []


for mode in sorted(
    df_mcdm['mode'].unique()
):

    mode_data = (
        df_mcdm[
            df_mcdm['mode'] == mode
        ]
        .sort_values(
            'passenger_count'
        )
    )

    saw_diff = (
        mode_data[
            'saw_score'
        ]
        .diff()
        .dropna()
    )

    rci_diff = (
        mode_data[
            'relative_condition_index'
        ]
        .diff()
        .dropna()
    )

    fare_diff = (
        mode_data[
            'raw_dynamic_fare'
        ]
        .diff()
        .dropna()
    )

    saw_monotonic = (
        saw_diff >= -1e-12
    ).all()

    rci_monotonic = (
        rci_diff >= -1e-12
    ).all()

    fare_monotonic = (
        fare_diff >= -1e-12
    ).all()

    monotonicity_results.append({
        'mode': mode,

        'demand_to_saw_monotonic': (
            saw_monotonic
        ),

        'demand_to_rci_monotonic': (
            rci_monotonic
        ),

        'demand_to_raw_fare_monotonic': (
            fare_monotonic
        ),

        'minimum_saw_difference': (
            saw_diff.min()
        ),

        'minimum_rci_difference': (
            rci_diff.min()
        ),

        'minimum_raw_fare_difference': (
            fare_diff.min()
        )
    })


monotonicity_df = pd.DataFrame(
    monotonicity_results
)


print(
    "\n=== DEMAND → SAW → RCI → FARE MONOTONICITY ==="
)

print(
    monotonicity_df
    .to_string(index=False)
)


if not (
    monotonicity_df[
        [
            'demand_to_saw_monotonic',
            'demand_to_rci_monotonic',
            'demand_to_raw_fare_monotonic'
        ]
    ]
    .all()
    .all()
):

    raise ValueError(
        "Monotonicity validation failed."
    )


print(
    "\n✓ Model monotonicity validated."
)


monotonicity_df.to_csv(
    'model_monotonicity_validation_2026.csv',
    index=False
)


# ============================================================
# 50. POLICY CONSTRAINT IMPLEMENTATION CHECK
# ============================================================

# Verify that final fare is exactly the mathematical clipping
# of raw fare to the policy interval.

expected_final_fare = (
    df_mcdm['raw_dynamic_fare']
    .clip(
        lower=df_mcdm['policy_min_fare'],
        upper=df_mcdm['policy_max_fare']
    )
)


policy_implementation_error = (
    (
        df_mcdm[
            'final_recommended_fare'
        ]
        - expected_final_fare
    )
    .abs()
    .max()
)


print(
    "\n=== POLICY CONSTRAINT IMPLEMENTATION CHECK ==="
)

print(
    f"Maximum clipping error: "
    f"{policy_implementation_error:.12f}"
)


if policy_implementation_error > 1e-9:

    raise ValueError(
        "Final fare does not match policy clipping."
    )


print(
    "\n✓ Policy constraint implementation validated."
)


# ============================================================
# 51. DEMAND PERTURBATION SENSITIVITY
# ============================================================

# ------------------------------------------------------------
# Purpose:
#
# Test model response when observed passenger counts contain
# small perturbations.
#
# Two scenarios:
#
#   ±5% random perturbation
#   ±10% random perturbation
#
# Random seed is fixed for reproducibility.
#
# IMPORTANT:
# These are sensitivity scenarios, NOT estimates of actual
# measurement error.
# ------------------------------------------------------------

N_PERTURBATION = 300

perturbation_rng = np.random.default_rng(
    2026
)

demand_perturbation_results = []


for noise_level in [
    0.05,
    0.10
]:

    for iteration in range(
        N_PERTURBATION
    ):

        perturbed_data = (
            df_final.copy()
        )

        # Uniform perturbation around the observed value
        multipliers = (
            perturbation_rng.uniform(
                1 - noise_level,
                1 + noise_level,
                size=len(
                    perturbed_data
                )
            )
        )

        perturbed_data[
            'passenger_count'
        ] = (
            perturbed_data[
                'passenger_count'
            ]
            * multipliers
        )

        # Recalculate normalization
        R_perturbed = (
            normalize_criteria(
                perturbed_data
            )
        )

        # Recalculate objective weights
        W_perturbed = (
            calculate_entropy_weights(
                R_perturbed
            )
        )


        # Recalculate SAW
        perturbed_score = (
            R_perturbed[
                criterion_cols
            ]
            .dot(
                W_perturbed[
                    criterion_cols
                ]
            )
        )


        # Mode-specific baseline
        perturbed_mode_baseline = (
            pd.DataFrame({
                'mode': perturbed_data[
                    'mode'
                ],
                'score': perturbed_score
            })
            .groupby('mode')['score']
            .transform('mean')
        )

        # RCI
        perturbed_rci = (
            perturbed_score
            / perturbed_mode_baseline
        )

        # Raw dynamic fare
        perturbed_raw_fare = (
            perturbed_data[
                'reference_fare'
            ]
            * (
                1
                + ALPHA
                * (
                    perturbed_rci
                    - 1
                )
            )
        )

        # Final policy-constrained fare
        perturbed_final_fare = (
            perturbed_raw_fare
            .clip(
                lower=perturbed_data[
                    'mode'
                ].map(
                    policy_min_fare
                ),
                upper=perturbed_data[
                    'mode'
                ].map(
                    policy_max_fare
                )
            )
        )

        # Save each mode summary
        for mode in sorted(
            perturbed_data[
                'mode'
            ].unique()
        ):

            mask = (
                perturbed_data[
                    'mode'
                ] == mode
            )

            demand_perturbation_results.append({
                'noise_level_pct': (
                    noise_level
                    * 100
                ),

                'iteration': (
                    iteration + 1
                ),

                'mode': mode,

                'mean_raw_fare': (
                    perturbed_raw_fare[
                        mask
                    ].mean()
                ),

                'min_raw_fare': (
                    perturbed_raw_fare[
                        mask
                    ].min()
                ),

                'max_raw_fare': (
                    perturbed_raw_fare[
                        mask
                    ].max()
                ),

                'mean_final_fare': (
                    perturbed_final_fare[
                        mask
                    ].mean()
                ),

                'min_final_fare': (
                    perturbed_final_fare[
                        mask
                    ].min()
                ),

                'max_final_fare': (
                    perturbed_final_fare[
                        mask
                    ].max()
                ),

                'C1_weight': (
                    W_perturbed[
                        'C1_demand_normalized'
                    ]
                ),

                'C2_weight': (
                    W_perturbed[
                        'C2_coverage_normalized'
                    ]
                ),

                'C3_weight': (
                    W_perturbed[
                        'C3_affordability_normalized'
                    ]
                )
            })


demand_perturbation_df = pd.DataFrame(
    demand_perturbation_results
)


# ------------------------------------------------------------
# Aggregate perturbation results
# ------------------------------------------------------------

demand_perturbation_summary = (
    demand_perturbation_df
    .groupby(
        [
            'noise_level_pct',
            'mode'
        ]
    )
    .agg(
        raw_fare_mean=(
            'mean_raw_fare',
            'mean'
        ),

        raw_fare_min=(
            'min_raw_fare',
            'min'
        ),

        raw_fare_max=(
            'max_raw_fare',
            'max'
        ),

        final_fare_mean=(
            'mean_final_fare',
            'mean'
        ),

        final_fare_min=(
            'min_final_fare',
            'min'
        ),

        final_fare_max=(
            'max_final_fare',
            'max'
        ),

        C1_weight_mean=(
            'C1_weight',
            'mean'
        ),

        C1_weight_std=(
            'C1_weight',
            'std'
        ),

        C2_weight_mean=(
            'C2_weight',
            'mean'
        ),

        C2_weight_std=(
            'C2_weight',
            'std'
        ),

        C3_weight_mean=(
            'C3_weight',
            'mean'
        ),

        C3_weight_std=(
            'C3_weight',
            'std'
        )
    )
    .reset_index()
)


print(
    "\n=== DEMAND PERTURBATION ROBUSTNESS ==="
)

print(
    demand_perturbation_summary
    .to_string(index=False)
)


demand_perturbation_df.to_csv(
    'demand_perturbation_results_2026.csv',
    index=False
)

demand_perturbation_summary.to_csv(
    'demand_perturbation_summary_2026.csv',
    index=False
)


# ============================================================
# 52. OVERALL ROBUSTNESS DASHBOARD
# ============================================================

# ------------------------------------------------------------
# Collect the most important robustness indicators in one
# compact table for the paper.
# ------------------------------------------------------------

overall_robustness = pd.DataFrame({
    'validation': [
        'Baseline entropy weight sum',
        'RCI mean deviation from 1',
        'Temporal weight max range',
        'Bootstrap maximum relative weight deviation',
        'Weight sensitivity minimum SAW Spearman',
        'Criterion exclusion minimum SAW Spearman',
        'Alpha sensitivity scenarios',
        'Monotonicity validation',
        'Policy clipping validation'
    ],

    'result': [
        baseline_weights.sum(),

        (
            df_mcdm
            .groupby('mode')[
                'relative_condition_index'
            ]
            .mean()
            .sub(1)
            .abs()
            .max()
        ),

        temporal_weight_summary[
            'range'
        ].max(),

        bootstrap_summary[
            'relative_mean_difference_pct'
        ].max(),

        weight_sensitivity_df[
            'SAW_spearman_vs_baseline'
        ].min(),

        criterion_exclusion_df[
            'SAW_spearman_vs_baseline'
        ].min(),

        len(
            alpha_validation_values
        ),

        bool(
            monotonicity_df[
                [
                    'demand_to_saw_monotonic',
                    'demand_to_rci_monotonic',
                    'demand_to_raw_fare_monotonic'
                ]
            ]
            .all()
            .all()
        ),

        policy_implementation_error
    ]
})


print(
    "\n=== OVERALL MODEL ROBUSTNESS DASHBOARD ==="
)

print(
    overall_robustness
    .to_string(index=False)
)


overall_robustness.to_csv(
    'model_robustness_dashboard_2026.csv',
    index=False
)


# ============================================================
# 53. FINAL VALIDATION MESSAGE
# ============================================================

print("\n")
print("=" * 70)
print("MODEL ROBUSTNESS ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nGenerated validation files:"
)

print(
    "1. temporal_entropy_weight_stability_2026.csv"
)

print(
    "2. temporal_saw_stability_2026.csv"
)

print(
    "3. bootstrap_entropy_weights_2026.csv"
)

print(
    "4. bootstrap_entropy_weight_summary_2026.csv"
)

print(
    "5. weight_sensitivity_2026.csv"
)

print(
    "6. criterion_exclusion_robustness_2026.csv"
)

print(
    "7. alpha_robustness_2026.csv"
)

print(
    "8. model_monotonicity_validation_2026.csv"
)

print(
    "9. demand_perturbation_results_2026.csv"
)

print(
    "10. demand_perturbation_summary_2026.csv"
)

print(
    "11. model_robustness_dashboard_2026.csv"
)

print(
    "\n✓ Robustness validation pipeline completed."
)







MODEL VALIDATION & ROBUSTNESS ANALYSIS

=== BASELINE WEIGHTS ===
C1_demand_normalized           0.051820
C2_coverage_normalized         0.799802
C3_affordability_normalized    0.148377

Weight sum: 1.000000

=== TEMPORAL ENTROPY WEIGHT STABILITY ===
 quarter  C1_demand_normalized  C2_coverage_normalized  C3_affordability_normalized
     1.0              0.056995                0.795437                     0.147567
     3.0              0.036097                0.813065                     0.150838
     4.0              0.032226                0.816331                     0.151444

=== QUARTERLY SAW RANK STABILITY ===
 quarter  observations  saw_spearman_vs_baseline
       1           270                       1.0
       3           183                       1.0
       4           276                       1.0

=== TEMPORAL WEIGHT SUMMARY ===
                  criterion  mean_weight  std_weight  min_weight  max_weight    range
       C1_demand_normalized     0.041773    0.013325    0.0

In [ ]:
# ============================================================
# 54. TEMPORAL HOLDOUT / WALK-FORWARD ROBUSTNESS
# ============================================================

# ------------------------------------------------------------
# OBJECTIVE
#
# Test whether model parameters derived from earlier periods
# remain consistent when applied to later, unseen periods.
#
# This is a robustness / temporal generalization test.
# It is NOT predictive accuracy validation because the study
# does not have an observed "true optimal fare" as ground truth.
#
# Method:
#   - Sort all observed dates chronologically
#   - Split observed dates into 5 chronological blocks
#   - Use expanding-window training
#   - Apply training-derived normalization + entropy weights
#     to the next unseen block
#   - Compare holdout SAW / fare results against the baseline
#     full-sample model on the same holdout observations
# ------------------------------------------------------------


print("\n")
print("=" * 70)
print("TEMPORAL HOLDOUT / WALK-FORWARD ROBUSTNESS")
print("=" * 70)


# ============================================================
# 54A. PREPARE CHRONOLOGICAL DATE BLOCKS
# ============================================================

N_BLOCKS = 5

observed_dates = (
    df_final['date']
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

date_blocks = np.array_split(
    observed_dates.values,
    N_BLOCKS
)

print("\n=== TEMPORAL BLOCKS ===")

for i, block in enumerate(
    date_blocks,
    start=1
):

    if len(block) == 0:
        continue

    print(
        f"Block {i}: "
        f"{pd.Timestamp(block[0]).date()} "
        f"to "
        f"{pd.Timestamp(block[-1]).date()} "
        f"({len(block)} dates)"
    )


# ============================================================
# 54B. NORMALIZATION PARAMETERS
# ============================================================

def fit_normalization_parameters(data):
    """
    Estimate normalization parameters from TRAIN data only.
    """

    # --------------------------------------------------------
    # C1 — mode-specific training peak demand
    # --------------------------------------------------------

    c1_peak_demand_by_mode = (
        data
        .groupby('mode')['passenger_count']
        .max()
    )

    if c1_peak_demand_by_mode.empty:

        raise ValueError(
            "Training data contains no valid mode-specific "
            "demand values."
        )

    if (
        c1_peak_demand_by_mode <= 0
    ).any():

        raise ValueError(
            "Training data contains invalid peak demand."
        )

    # --------------------------------------------------------
    # C2 — population access coverage
    # --------------------------------------------------------

    c2_max_population_access_coverage = (
        data[
            'population_access_coverage'
        ]
        .max()
    )

    if c2_max_population_access_coverage <= 0:

        raise ValueError(
            "Training data contains invalid population access "
            "network coverage."
        )

    # --------------------------------------------------------
    # C3 — affordability burden
    # --------------------------------------------------------

    c3_min_burden = (
        data[
            'fare_affordability_burden'
        ]
        .min()
    )

    if c3_min_burden <= 0:

        raise ValueError(
            "Training data contains invalid affordability "
            "burden."
        )

    return {

        'C1_peak_demand_by_mode':
            c1_peak_demand_by_mode,

        'C2_max_population_access_coverage':
            c2_max_population_access_coverage,

        'C3_min_burden':
            c3_min_burden

    }


# ============================================================
# 54B. APPLY TRAIN NORMALIZATION
# ============================================================

def apply_train_normalization(
    data,
    params,
    validate_training_range=False
):
    """
    Apply normalization parameters learned from TRAIN data.

    Important:
    - TRAIN data should remain within [0, 1].
    - TEST/HOLDOUT data may exceed [0, 1] when its value
      exceeds the maximum observed in TRAIN.
    - Test values are NOT clipped because doing so would hide
      genuine temporal changes.
    """

    result = pd.DataFrame(
        index=data.index
    )


    # --------------------------------------------------------
    # C1 — Mode-specific peak demand
    # --------------------------------------------------------

    c1_denominator = (
        data['mode']
        .map(
            params[
                'C1_peak_demand_by_mode'
            ]
        )
    )


    if c1_denominator.isna().any():

        missing_modes = (
            data.loc[
                c1_denominator.isna(),
                'mode'
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "Missing C1 training peak for modes: "
            f"{missing_modes}"
        )


    result[
        'C1_demand_normalized'
    ] = (
        data['passenger_count']
        /
        c1_denominator
    )


    # --------------------------------------------------------
    # C2 — Population Access Coverage
    # --------------------------------------------------------

    result[
        'C2_coverage_normalized'
    ] = (
        data[
            'population_access_coverage'
        ]
        /
        params[
            'C2_max_population_access_coverage'
        ]
    )


    # --------------------------------------------------------
    # C3 — Fare Affordability Burden
    # --------------------------------------------------------

    result[
        'C3_affordability_normalized'
    ] = (
        params[
            'C3_min_burden'
        ]
        /
        data[
            'fare_affordability_burden'
        ]
    )


    # --------------------------------------------------------
    # Basic numerical validation
    # --------------------------------------------------------

    criterion_cols_local = [
        'C1_demand_normalized',
        'C2_coverage_normalized',
        'C3_affordability_normalized'
    ]


    if (
        result[
            criterion_cols_local
        ]
        .isna()
        .any()
        .any()
    ):

        raise ValueError(
            "Train-based normalization produced "
            "missing values."
        )


    if (
        ~np.isfinite(
            result[
                criterion_cols_local
            ]
            .to_numpy()
        )
    ).any():

        raise ValueError(
            "Train-based normalization produced "
            "non-finite values."
        )


    # --------------------------------------------------------
    # Range validation
    #
    # Only TRAIN data is required to be within [0, 1].
    # HOLDOUT data is allowed to exceed 1 when the observed
    # value is larger than the training reference.
    # --------------------------------------------------------

    if validate_training_range:

        outside_range = (
            ~result[
                criterion_cols_local
            ]
            .apply(
                lambda col: col.between(
                    0,
                    1
                )
            )
            .all()
            .all()
        )


        if outside_range:

            raise ValueError(
                "Training normalization produced "
                "values outside [0, 1]."
            )


    return result


# ============================================================
# 54C. BASELINE REFERENCE FOR COMPARISON
# ============================================================

# The baseline model is calculated using the complete observed
# dataset. It is used only as a reference benchmark.
#
# baseline_rci is explicitly named to distinguish it from the
# holdout RCI calculated using training-period parameters.


baseline_reference = (
    df_mcdm[
        [
            'date',
            'mode',
            'saw_score',
            'relative_condition_index',
            'raw_dynamic_fare',
            'final_recommended_fare'
        ]
    ]
    .rename(
        columns={
            'relative_condition_index':
                'baseline_rci'
        }
    )
    .copy()
)


# ============================================================
# 54D. WALK-FORWARD VALIDATION
# ============================================================

walkforward_results = []

walkforward_observations = []


# ------------------------------------------------------------
# Block 1 cannot be evaluated because there is no earlier
# training period.
#
# Therefore:
#
#   Train = Block 1
#   Test  = Block 2
#
#   Train = Block 1 + Block 2
#   Test  = Block 3
#
#   Train = Block 1 + Block 2 + Block 3
#   Test  = Block 4
#
#   Train = Block 1 + ... + Block 4
#   Test  = Block 5
# ------------------------------------------------------------

for test_block_number in range(
    1,
    N_BLOCKS
):

    train_blocks = date_blocks[
        :test_block_number
    ]

    test_dates = date_blocks[
        test_block_number
    ]

    train_dates = np.concatenate(
        train_blocks
    )

    train_date_set = set(
        pd.to_datetime(train_dates)
    )

    test_date_set = set(
        pd.to_datetime(test_dates)
    )

    train_data = (
        df_final[
            df_final['date']
            .isin(train_date_set)
        ]
        .copy()
    )

    test_data = (
        df_final[
            df_final['date']
            .isin(test_date_set)
        ]
        .copy()
    )


    # ========================================================
    # FIT ON TRAIN ONLY
    # ========================================================

    normalization_params = (
        fit_normalization_parameters(
            train_data
        )
    )

    R_train = (
    apply_train_normalization(
        train_data,
        normalization_params,
        validate_training_range=True
        )
    )

    train_weights = (
        calculate_entropy_weights(
            R_train
        )
    )


    # ========================================================
    # APPLY TRAIN PARAMETERS TO TEST
    # ========================================================

    R_test = (
    apply_train_normalization(
        test_data,
        normalization_params,
        validate_training_range=False
        )
    )


    # ========================================================
    # TEST NORMALIZATION DIAGNOSTICS
    # ========================================================

    print(
        "\n=== HOLDOUT NORMALIZATION RANGE ==="
    )

    for col in criterion_cols:

        col_values = R_test[col]

        print(
            f"{col}: "
            f"min={col_values.min():.6f}, "
            f"max={col_values.max():.6f}, "
            f">1={(col_values > 1).sum()}, "
            f"<0={(col_values < 0).sum()}"
        )


    # ========================================================
    # TRAIN SAW
    # ========================================================

    train_saw = (
        R_train[
            criterion_cols
        ]
        .dot(
            train_weights[
                criterion_cols
            ]
        )
    )

    train_score_reference = pd.DataFrame({
        'mode': train_data['mode'].values,
        'train_saw': train_saw.values
    })

    train_mode_baseline = (
        train_score_reference
        .groupby('mode')['train_saw']
        .mean()
    )


    # ========================================================
    # TEST SAW
    # ========================================================

    test_saw = (
        R_test[
            criterion_cols
        ]
        .dot(
            train_weights[
                criterion_cols
            ]
        )
    )


    # ========================================================
    # TEST RCI
    #
    # IMPORTANT:
    # Baseline comes from TRAIN data only.
    #
    # This avoids using the test period itself to establish
    # the reference condition.
    # ========================================================

    test_baseline = (
        test_data['mode']
        .map(
            train_mode_baseline
        )
    )

    test_rci = (
        test_saw
        / test_baseline.values
    )


    # ========================================================
    # TEST RAW DYNAMIC FARE
    # ========================================================

    test_raw_fare = (
        test_data['reference_fare']
        .values
        * (
            1
            + ALPHA
            * (
                test_rci.values
                - 1
            )
        )
    )


    # ========================================================
    # TEST POLICY-CONSTRAINED FARE
    # ========================================================

    test_min_policy = (
        test_data['mode']
        .map(
            policy_min_fare
        )
        .values
    )

    test_max_policy = (
        test_data['mode']
        .map(
            policy_max_fare
        )
        .values
    )

    test_final_fare = np.clip(
        test_raw_fare,
        test_min_policy,
        test_max_policy
    )


    # ========================================================
    # BUILD HOLDOUT RESULT TABLE
    # ========================================================

    holdout_result = test_data[
        [
            'date',
            'mode',
            'reference_fare'
        ]
    ].copy()

    holdout_result[
        'holdout_saw_score'
    ] = test_saw.values

    holdout_result[
        'holdout_rci'
    ] = test_rci.values

    holdout_result[
        'holdout_raw_dynamic_fare'
    ] = test_raw_fare

    holdout_result[
        'holdout_final_fare'
    ] = test_final_fare

    holdout_result[
        'train_baseline_saw'
    ] = (
        test_baseline.values
    )

    holdout_result[
        'test_block'
    ] = (
        test_block_number + 1
    )

    holdout_result[
        'train_blocks'
    ] = (
        test_block_number
    )


    # ========================================================
    # MERGE BASELINE FULL-DATA MODEL
    # ========================================================

    holdout_result = holdout_result.merge(
        baseline_reference,
        on=[
            'date',
            'mode'
        ],
        how='left'
    )


    # ========================================================
    # SAW RANK CONSISTENCY
    # ========================================================

    saw_spearman = (
        holdout_result[
            'holdout_saw_score'
        ]
        .corr(
            holdout_result[
                'saw_score'
            ],
            method='spearman'
        )
    )


    # ========================================================
    # RAW FARE DIFFERENCE
    # ========================================================

    raw_fare_difference_pct = (
        (
            holdout_result[
                'holdout_raw_dynamic_fare'
            ]
            - holdout_result[
                'raw_dynamic_fare'
            ]
        )
        .abs()
        /
        holdout_result[
            'reference_fare'
        ]
        * 100
    )


    # ========================================================
    # FINAL FARE DIFFERENCE
    # ========================================================

    final_fare_difference_pct = (
        (
            holdout_result[
                'holdout_final_fare'
            ]
            - holdout_result[
                'final_recommended_fare'
            ]
        )
        .abs()
        /
        holdout_result[
            'reference_fare'
        ]
        * 100
    )


    # ========================================================
    # RAW PRICING DIRECTION AGREEMENT
    # ========================================================

    baseline_raw_direction = np.select(
        [
            holdout_result[
                'raw_dynamic_fare'
            ]
            >
            holdout_result[
                'reference_fare'
            ],

            holdout_result[
                'raw_dynamic_fare'
            ]
            <
            holdout_result[
                'reference_fare'
            ]
        ],
        [
            'increase',
            'decrease'
        ],
        default='no_change'
    )

    holdout_raw_direction = np.select(
        [
            holdout_result[
                'holdout_raw_dynamic_fare'
            ]
            >
            holdout_result[
                'reference_fare'
            ],

            holdout_result[
                'holdout_raw_dynamic_fare'
            ]
            <
            holdout_result[
                'reference_fare'
            ]
        ],
        [
            'increase',
            'decrease'
        ],
        default='no_change'
    )

    direction_agreement = (
        baseline_raw_direction
        ==
        holdout_raw_direction
    ).mean()


    # ========================================================
    # STORE OBSERVATION-LEVEL RESULTS
    # ========================================================

    walkforward_observations.append(
        holdout_result
    )


    # ========================================================
    # STORE BLOCK SUMMARY
    # ========================================================

    walkforward_results.append({
        'test_block': (
            test_block_number + 1
        ),

        'train_start': (
            train_data['date']
            .min()
        ),

        'train_end': (
            train_data['date']
            .max()
        ),

        'test_start': (
            test_data['date']
            .min()
        ),

        'test_end': (
            test_data['date']
            .max()
        ),

        'train_observations': (
            len(train_data)
        ),

        'test_observations': (
            len(test_data)
        ),

        'C1_weight': (
            train_weights[
                'C1_demand_normalized'
            ]
        ),

        'C2_weight': (
            train_weights[
                'C2_coverage_normalized'
            ]
        ),

        'C3_weight': (
            train_weights[
                'C3_affordability_normalized'
            ]
        ),

        'C1_abs_difference_vs_baseline': (
            abs(
                train_weights[
                    'C1_demand_normalized'
                ]
                -
                baseline_weights[
                    'C1_demand_normalized'
                ]
            )
        ),

        'C2_abs_difference_vs_baseline': (
            abs(
                train_weights[
                    'C2_coverage_normalized'
                ]
                -
                baseline_weights[
                    'C2_coverage_normalized'
                ]
            )
        ),

        'C3_abs_difference_vs_baseline': (
            abs(
                train_weights[
                    'C3_affordability_normalized'
                ]
                -
                baseline_weights[
                    'C3_affordability_normalized'
                ]
            )
        ),

        'SAW_spearman_vs_full_model': (
            saw_spearman
        ),

        'mean_absolute_raw_fare_difference_pct': (
            raw_fare_difference_pct.mean()
        ),

        'max_absolute_raw_fare_difference_pct': (
            raw_fare_difference_pct.max()
        ),

        'mean_absolute_final_fare_difference_pct': (
            final_fare_difference_pct.mean()
        ),

        'max_absolute_final_fare_difference_pct': (
            final_fare_difference_pct.max()
        ),

        'raw_pricing_direction_agreement_pct': (
            direction_agreement
            * 100
        )
    })


# ============================================================
# 54E. COMBINE RESULTS
# ============================================================

walkforward_summary = (
    pd.DataFrame(
        walkforward_results
    )
)

walkforward_detail = (
    pd.concat(
        walkforward_observations,
        ignore_index=True
    )
)


# ============================================================
# 54F. DISPLAY RESULTS
# ============================================================

print(
    "\n=== WALK-FORWARD VALIDATION SUMMARY ==="
)

print(
    walkforward_summary
    .to_string(index=False)
)


# ============================================================
# 54G. AGGREGATE ROBUSTNESS STATISTICS
# ============================================================

walkforward_aggregate = pd.DataFrame({
    'metric': [
        'Number of evaluated holdout blocks',

        'Minimum SAW Spearman correlation',

        'Mean SAW Spearman correlation',

        'Maximum SAW Spearman correlation',

        'Maximum C1 weight deviation',

        'Maximum C2 weight deviation',

        'Maximum C3 weight deviation',

        'Maximum mean raw fare difference (%)',

        'Maximum mean final fare difference (%)',

        'Minimum raw pricing direction agreement (%)'
    ],

    'value': [
        len(
            walkforward_summary
        ),

        walkforward_summary[
            'SAW_spearman_vs_full_model'
        ].min(),

        walkforward_summary[
            'SAW_spearman_vs_full_model'
        ].mean(),

        walkforward_summary[
            'SAW_spearman_vs_full_model'
        ].max(),

        walkforward_summary[
            'C1_abs_difference_vs_baseline'
        ].max(),

        walkforward_summary[
            'C2_abs_difference_vs_baseline'
        ].max(),

        walkforward_summary[
            'C3_abs_difference_vs_baseline'
        ].max(),

        walkforward_summary[
            'mean_absolute_raw_fare_difference_pct'
        ].max(),

        walkforward_summary[
            'mean_absolute_final_fare_difference_pct'
        ].max(),

        walkforward_summary[
            'raw_pricing_direction_agreement_pct'
        ].min()
    ]
})


print(
    "\n=== WALK-FORWARD AGGREGATE ROBUSTNESS ==="
)

print(
    walkforward_aggregate
    .to_string(index=False)
)


# ============================================================
# 54H. HOLDOUT RCI SUMMARY
# ============================================================

rci_holdout_summary = (
    walkforward_detail
    .groupby('mode')
    .agg(
        min_holdout_rci=(
            'holdout_rci',
            'min'
        ),

        mean_holdout_rci=(
            'holdout_rci',
            'mean'
        ),

        max_holdout_rci=(
            'holdout_rci',
            'max'
        )
    )
    .reset_index()
)

print(
    "\n=== HOLDOUT RCI SUMMARY ==="
)

print(
    rci_holdout_summary
    .to_string(index=False)
)


# ============================================================
# 54I. HOLDOUT FARE SUMMARY
# ============================================================

holdout_fare_summary = (
    walkforward_detail
    .groupby('mode')
    .agg(
        min_holdout_raw_fare=(
            'holdout_raw_dynamic_fare',
            'min'
        ),

        max_holdout_raw_fare=(
            'holdout_raw_dynamic_fare',
            'max'
        ),

        mean_holdout_raw_fare=(
            'holdout_raw_dynamic_fare',
            'mean'
        ),

        min_holdout_final_fare=(
            'holdout_final_fare',
            'min'
        ),

        max_holdout_final_fare=(
            'holdout_final_fare',
            'max'
        ),

        mean_holdout_final_fare=(
            'holdout_final_fare',
            'mean'
        )
    )
    .reset_index()
)

print(
    "\n=== HOLDOUT FARE SUMMARY ==="
)

print(
    holdout_fare_summary
    .to_string(index=False)
)


# ============================================================
# 54J. VALIDATE HOLDOUT RESULTS
# ============================================================

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

holdout_required_cols = [
    'holdout_saw_score',
    'holdout_rci',
    'holdout_raw_dynamic_fare',
    'holdout_final_fare',
    'train_baseline_saw'
]

holdout_missing = (
    walkforward_detail[
        holdout_required_cols
    ]
    .isna()
    .sum()
)

print(
    "\n=== HOLDOUT MISSING VALUE VALIDATION ==="
)

print(
    holdout_missing
)

if holdout_missing.sum() > 0:
    raise ValueError(
        "Temporal holdout contains missing values."
    )


# ------------------------------------------------------------
# Policy feasibility
# ------------------------------------------------------------

holdout_lower_violations = (
    walkforward_detail[
        'holdout_final_fare'
    ]
    <
    walkforward_detail[
        'mode'
    ].map(
        policy_min_fare
    ).values
).sum()

holdout_upper_violations = (
    walkforward_detail[
        'holdout_final_fare'
    ]
    >
    walkforward_detail[
        'mode'
    ].map(
        policy_max_fare
    ).values
).sum()


print(
    f"\nHoldout lower-bound violations: "
    f"{holdout_lower_violations}"
)

print(
    f"Holdout upper-bound violations: "
    f"{holdout_upper_violations}"
)

if (
    holdout_lower_violations > 0
    or
    holdout_upper_violations > 0
):
    raise ValueError(
        "Holdout final fares violate policy constraints."
    )


print(
    "\n✓ Temporal holdout policy constraints validated."
)


# ============================================================
# 54K. SAVE WALK-FORWARD RESULTS
# ============================================================

walkforward_summary.to_csv(
    'temporal_holdout_summary_2026.csv',
    index=False
)

walkforward_detail.to_csv(
    'temporal_holdout_detail_2026.csv',
    index=False
)

walkforward_aggregate.to_csv(
    'temporal_holdout_aggregate_2026.csv',
    index=False
)

rci_holdout_summary.to_csv(
    'temporal_holdout_rci_summary_2026.csv',
    index=False
)

holdout_fare_summary.to_csv(
    'temporal_holdout_fare_summary_2026.csv',
    index=False
)


# ============================================================
# 54L. FINAL MESSAGE
# ============================================================

print("\n")
print("=" * 70)
print("TEMPORAL HOLDOUT VALIDATION COMPLETED")
print("=" * 70)

print(
    "\nGenerated files:"
)

print(
    "1. temporal_holdout_summary_2026.csv"
)

print(
    "2. temporal_holdout_detail_2026.csv"
)

print(
    "3. temporal_holdout_aggregate_2026.csv"
)

print(
    "4. temporal_holdout_rci_summary_2026.csv"
)

print(
    "5. temporal_holdout_fare_summary_2026.csv"
)

print(
    "\n✓ Walk-forward robustness analysis completed."
)






TEMPORAL HOLDOUT / WALK-FORWARD ROBUSTNESS

=== TEMPORAL BLOCKS ===
Block 1: 2023-01-01 to 2023-02-18 (49 dates)
Block 2: 2023-02-19 to 2023-08-08 (49 dates)
Block 3: 2023-08-09 to 2023-09-26 (49 dates)
Block 4: 2023-09-27 to 2023-11-13 (48 dates)
Block 5: 2023-11-14 to 2023-12-31 (48 dates)

=== HOLDOUT NORMALIZATION RANGE ===
C1_demand_normalized: min=0.028602, max=1.195164, >1=28, <0=0
C2_coverage_normalized: min=0.055966, max=1.000000, >1=0, <0=0
C3_affordability_normalized: min=0.250000, max=1.000000, >1=0, <0=0

=== HOLDOUT NORMALIZATION RANGE ===
C1_demand_normalized: min=0.243956, max=1.093520, >1=23, <0=0
C2_coverage_normalized: min=0.055966, max=1.000000, >1=0, <0=0
C3_affordability_normalized: min=0.250000, max=1.000000, >1=0, <0=0

=== HOLDOUT NORMALIZATION RANGE ===
C1_demand_normalized: min=0.473410, max=1.031754, >1=20, <0=0
C2_coverage_normalized: min=0.055966, max=1.000000, >1=0, <0=0
C3_affordability_normalized: min=0.250000, max=1.000000, >1=0, <0=0

=== HOLDOUT NO

In [ ]:
# ============================================================
# 55. WITHIN-MODE TEMPORAL HOLDOUT VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("WITHIN-MODE TEMPORAL HOLDOUT VALIDATION")
print("=" * 70)


within_mode_results = []


for mode in sorted(
    walkforward_detail['mode'].unique()
):

    mode_data = (
        walkforward_detail[
            walkforward_detail['mode'] == mode
        ]
        .copy()
    )

    # ========================================================
    # 55A. RCI STABILITY
    # ========================================================

    rci_spearman = (
        mode_data[
            'holdout_rci'
        ]
        .corr(
            mode_data[
                'baseline_rci'
            ],
            method='spearman'
        )
    )

    # ========================================================
    # 55B. RAW FARE STABILITY
    # ========================================================

    raw_fare_spearman = (
        mode_data[
            'holdout_raw_dynamic_fare'
        ]
        .corr(
            mode_data[
                'raw_dynamic_fare'
            ],
            method='spearman'
        )
    )

    # ========================================================
    # 55C. RCI ABSOLUTE DIFFERENCE
    # ========================================================

    rci_abs_difference = (
        (
            mode_data[
                'holdout_rci'
            ]
            -
            mode_data[
                'baseline_rci'
            ]
        )
        .abs()
    )

    mean_abs_rci_difference = (
        rci_abs_difference.mean()
    )

    max_abs_rci_difference = (
        rci_abs_difference.max()
    )

    # ========================================================
    # 55D. RAW FARE ABSOLUTE DIFFERENCE
    # ========================================================

    raw_fare_abs_difference = (
        (
            mode_data[
                'holdout_raw_dynamic_fare'
            ]
            -
            mode_data[
                'raw_dynamic_fare'
            ]
        )
        .abs()
    )

    mean_abs_raw_fare_difference = (
        raw_fare_abs_difference.mean()
    )

    max_abs_raw_fare_difference = (
        raw_fare_abs_difference.max()
    )

    # ========================================================
    # 55E. RAW FARE PERCENTAGE DIFFERENCE
    # ========================================================

    raw_fare_pct_difference = (
        raw_fare_abs_difference
        /
        mode_data[
            'reference_fare'
        ]
        * 100
    )

    mean_abs_raw_fare_difference_pct = (
        raw_fare_pct_difference.mean()
    )

    max_abs_raw_fare_difference_pct = (
        raw_fare_pct_difference.max()
    )

    # ========================================================
    # 55F. FINAL FARE ABSOLUTE DIFFERENCE
    # ========================================================

    final_fare_abs_difference = (
        (
            mode_data[
                'holdout_final_fare'
            ]
            -
            mode_data[
                'final_recommended_fare'
            ]
        )
        .abs()
    )

    mean_abs_final_fare_difference = (
        final_fare_abs_difference.mean()
    )

    max_abs_final_fare_difference = (
        final_fare_abs_difference.max()
    )

    # ========================================================
    # 55G. FINAL FARE PERCENTAGE DIFFERENCE
    # ========================================================

    final_fare_pct_difference = (
        final_fare_abs_difference
        /
        mode_data[
            'reference_fare'
        ]
        * 100
    )

    mean_abs_final_fare_difference_pct = (
        final_fare_pct_difference.mean()
    )

    max_abs_final_fare_difference_pct = (
        final_fare_pct_difference.max()
    )

    # ========================================================
    # 55H. RAW PRICING DIRECTION AGREEMENT
    # ========================================================

    baseline_direction = np.select(
        [
            mode_data[
                'raw_dynamic_fare'
            ]
            >
            mode_data[
                'reference_fare'
            ],

            mode_data[
                'raw_dynamic_fare'
            ]
            <
            mode_data[
                'reference_fare'
            ]
        ],
        [
            'increase',
            'decrease'
        ],
        default='no_change'
    )

    holdout_direction = np.select(
        [
            mode_data[
                'holdout_raw_dynamic_fare'
            ]
            >
            mode_data[
                'reference_fare'
            ],

            mode_data[
                'holdout_raw_dynamic_fare'
            ]
            <
            mode_data[
                'reference_fare'
            ]
        ],
        [
            'increase',
            'decrease'
        ],
        default='no_change'
    )

    direction_agreement = (
        (
            baseline_direction
            ==
            holdout_direction
        )
        .mean()
        * 100
    )

    # ========================================================
    # 55I. STORE RESULTS
    # ========================================================

    within_mode_results.append({
        'mode': mode,

        'observations': len(
            mode_data
        ),

        'RCI_spearman': (
            rci_spearman
        ),

        'raw_fare_spearman': (
            raw_fare_spearman
        ),

        'mean_absolute_RCI_difference': (
            mean_abs_rci_difference
        ),

        'max_absolute_RCI_difference': (
            max_abs_rci_difference
        ),

        'mean_absolute_raw_fare_difference': (
            mean_abs_raw_fare_difference
        ),

        'max_absolute_raw_fare_difference': (
            max_abs_raw_fare_difference
        ),

        'mean_absolute_raw_fare_difference_pct': (
            mean_abs_raw_fare_difference_pct
        ),

        'max_absolute_raw_fare_difference_pct': (
            max_abs_raw_fare_difference_pct
        ),

        'mean_absolute_final_fare_difference': (
            mean_abs_final_fare_difference
        ),

        'max_absolute_final_fare_difference': (
            max_abs_final_fare_difference
        ),

        'mean_absolute_final_fare_difference_pct': (
            mean_abs_final_fare_difference_pct
        ),

        'max_absolute_final_fare_difference_pct': (
            max_abs_final_fare_difference_pct
        ),

        'raw_pricing_direction_agreement_pct': (
            direction_agreement
        )
    })


# ============================================================
# 55J. BUILD RESULT DATAFRAME
# ============================================================

within_mode_df = pd.DataFrame(
    within_mode_results
)


# ============================================================
# 55K. DISPLAY
# ============================================================

print(
    "\n=== WITHIN-MODE TEMPORAL ROBUSTNESS ==="
)

print(
    within_mode_df
    .to_string(index=False)
)


# ============================================================
# 55L. AGGREGATE ROBUSTNESS
# ============================================================

within_mode_aggregate = pd.DataFrame({
    'metric': [
        'Minimum RCI Spearman',
        'Minimum raw fare Spearman',
        'Maximum mean raw fare difference (%)',
        'Maximum mean final fare difference (%)',
        'Minimum raw pricing direction agreement (%)'
    ],

    'value': [
        within_mode_df[
            'RCI_spearman'
        ].min(),

        within_mode_df[
            'raw_fare_spearman'
        ].min(),

        within_mode_df[
            'mean_absolute_raw_fare_difference_pct'
        ].max(),

        within_mode_df[
            'mean_absolute_final_fare_difference_pct'
        ].max(),

        within_mode_df[
            'raw_pricing_direction_agreement_pct'
        ].min()
    ]
})


print(
    "\n=== WITHIN-MODE AGGREGATE ==="
)

print(
    within_mode_aggregate
    .to_string(index=False)
)


# ============================================================
# 55M. VALIDATION CHECK
# ============================================================

required_baseline_cols = [
    'baseline_rci',
    'raw_dynamic_fare',
    'final_recommended_fare'
]

missing_baseline_cols = (
    [
        col
        for col in required_baseline_cols
        if col not in walkforward_detail.columns
    ]
)

if missing_baseline_cols:

    raise ValueError(
        "Within-mode baseline columns missing: "
        + str(missing_baseline_cols)
    )


if within_mode_df.isna().any().any():

    raise ValueError(
        "Within-mode robustness results contain NaN."
    )


print(
    "\n✓ Within-mode validation inputs and outputs are valid."
)


# ============================================================
# 55N. SAVE RESULTS
# ============================================================

within_mode_df.to_csv(
    'within_mode_temporal_robustness_2026.csv',
    index=False
)

within_mode_aggregate.to_csv(
    'within_mode_temporal_robustness_summary_2026.csv',
    index=False
)


print(
    "\n✓ Within-mode temporal robustness completed."
)



WITHIN-MODE TEMPORAL HOLDOUT VALIDATION

=== WITHIN-MODE TEMPORAL ROBUSTNESS ===
        mode  observations  RCI_spearman  raw_fare_spearman  mean_absolute_RCI_difference  max_absolute_RCI_difference  mean_absolute_raw_fare_difference  max_absolute_raw_fare_difference  mean_absolute_raw_fare_difference_pct  max_absolute_raw_fare_difference_pct  mean_absolute_final_fare_difference  max_absolute_final_fare_difference  mean_absolute_final_fare_difference_pct  max_absolute_final_fare_difference_pct  raw_pricing_direction_agreement_pct
         lrt           194      0.955675           0.955675                      0.013598                     0.037337                           6.799087                         18.668446                               0.135982                              0.373369                             0.000000                            0.000000                                 0.000000                                0.000000                            80.412371
     

In [ ]:
# ============================================================
# DEBUG C1 — FIND INVALID PASSENGER COUNT VALUES
# ============================================================

# Reload original demand data so we can inspect the raw values
df_debug = pd.read_csv(
    'Filedata Jumlah Penumpang Angkutan Umum yang Terlayani per Hari Tahun 2023.csv'
)

df_debug['jenis_moda'] = (
    df_debug['jenis_moda']
    .astype(str)
    .str.strip()
    .str.lower()
)

df_debug = df_debug[
    df_debug['jenis_moda'].isin([
        'transjakarta',
        'mrt',
        'lrt'
    ])
].copy()

# Preserve original value
df_debug['passenger_raw'] = (
    df_debug['jumlah_penumpang_per_hari']
)

# Try numeric conversion
df_debug['passenger_numeric'] = pd.to_numeric(
    df_debug['passenger_raw'],
    errors='coerce'
)

# Show rows that failed conversion
invalid_passenger = df_debug[
    df_debug['passenger_numeric'].isna()
].copy()

print("=== INVALID PASSENGER COUNT VALUES ===")
print(
    f"Total invalid values: {len(invalid_passenger)}"
)

print(
    invalid_passenger[
        [
            'tanggal',
            'jenis_moda',
            'passenger_raw'
        ]
    ]
    .to_string(index=False)
)

print("\n=== RAW VALUE FREQUENCY ===")
print(
    invalid_passenger['passenger_raw']
    .value_counts(dropna=False)
    .to_string()
)

=== INVALID PASSENGER COUNT VALUES ===
Total invalid values: 84
   tanggal   jenis_moda passenger_raw
2023-02-01          mrt       98,351 
2023-02-01          lrt        2,372 
2023-02-01 transjakarta      814,881 
2023-02-02          mrt       98,149 
2023-02-02          lrt        2,524 
2023-02-02 transjakarta      818,742 
2023-02-03          mrt      100,461 
2023-02-03          lrt        2,702 
2023-02-03 transjakarta      819,001 
2023-02-04          mrt       61,084 
2023-02-04          lrt        2,859 
2023-02-04 transjakarta      556,834 
2023-02-05          mrt       56,633 
2023-02-05          lrt        2,427 
2023-02-05 transjakarta      488,204 
2023-02-06          mrt       92,186 
2023-02-06          lrt        2,613 
2023-02-06 transjakarta      825,070 
2023-02-07          mrt       98,820 
2023-02-07          lrt        2,862 
2023-02-07 transjakarta      832,484 
2023-02-08          mrt       99,829 
2023-02-08          lrt        3,105 
2023-02-08 transjakarta 

In [ ]:
# from google.colab import files

# files.download('mcdm_saw_results_2026.csv')